# RideBase 05 — ML Preprocessing & Feature Preparation

Bu notebookun amacı RideBase Synthetic Dataset v1.2 verilerini, sonraki ML modellerinin güvenli şekilde kullanabileceği hale getirmektir.

**Bu notebookta model eğitilmeyecektir.** Hazırlanan veri ve kurallar şu üç çalışma için kullanılacaktır:

1. V1 Statistical Baseline
2. V2 Survival / Time-to-Event Model
3. V3 Next-Task Multi-Label Model

Buradaki sonuçlar sentetik v1.2 ML PoC pipeline'ına aittir. Production validation yapılmış değildir; production feasibility hâlâ `BLOCKED` durumundadır.

**Temel kavramlar:** Bir **feature**, modelin tahmin yaparken bildiği girdidir. Bir **target**, modelin öğrenmeye veya tahmin etmeye çalıştığı sonuçtur. **Preprocessing**, ham feature'ları modelin güvenli biçimde kullanabileceği sayısal yapıya hazırlama sürecidir.


## 1. Dataset version guard ve merkezi path

### Ne yapıyoruz?
Dataset klasörünü notebook konumuna göre buluyor; metadata içindeki sürüm, generator sürümü, kalite kapısı ve temel satır sayılarını kontrol ediyoruz. **Version guard**, yalnız beklediğimiz veri sürümüyle çalıştığımızı doğrulayan güvenlik kontrolüdür.

### Neden yapıyoruz?
v1.1 ve v1.2 aynı kolon adına sahip olsa bile kolonun anlamı, üretim kuralı veya satır sayısı değişmiş olabilir. Tek pipeline içinde sürümleri karıştırmak, görünmeyen veri uyumsuzluğu ve tekrar üretilemeyen sonuç yaratır.

### Yanlış yaparsak ne olur?
Eski veriden öğrenilen doldurma veya kategori kuralları yeni veriye uygulanabilir; model hatası preprocessing hatasıymış gibi görünmeyebilir.

### Bu dataset için kararımız
Yalnız repository kökündeki immutable `ridebase_v1_2/` okunacaktır. `ridebase-ml/data/` altındaki legacy kopyalar kullanılmayacaktır. Sürüm `1.2.0` değilse notebook duracaktır.


In [1]:
from pathlib import Path
import hashlib
import json
import math
import re

import joblib
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import numpy as np
import pandas as pd
import seaborn as sns
from IPython.display import display, Markdown
from scipy import sparse
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

pd.set_option("display.max_columns", 180)
pd.set_option("display.max_rows", 180)
pd.set_option("display.max_colwidth", 180)
sns.set_theme(style="whitegrid", context="notebook")


def find_project_root() -> Path:
    current = Path.cwd().resolve()
    for candidate in [current, *current.parents]:
        if (candidate / "notebooks").is_dir() and (candidate / "src").is_dir():
            return candidate
    raise FileNotFoundError("ridebase-ml proje kökü bulunamadı")


PROJECT_ROOT = find_project_root()
DATASET_ROOT = PROJECT_ROOT.parent / "ridebase_v1_2"
SOURCE_DIR = DATASET_ROOT / "source_tables"
DERIVED_DIR = DATASET_ROOT / "derived_outputs"
REPORTS_DIR = PROJECT_ROOT / "reports"
TABLES_DIR = REPORTS_DIR / "tables"
FIGURES_DIR = REPORTS_DIR / "figures" / "ml_preprocessing"
MODELS_DIR = PROJECT_ROOT / "models"
OUTPUTS_DIR = PROJECT_ROOT / "outputs"
DATASET_VERSION = "1.2.0"
PREPROCESSING_VERSION = "ML_PREPROCESSING_V1_2_1.0.0"
NEAR_CONSTANT_THRESHOLD = 0.99
RARE_CATEGORY_MIN_COUNT = 50

for directory in [TABLES_DIR, FIGURES_DIR, MODELS_DIR, OUTPUTS_DIR]:
    directory.mkdir(parents=True, exist_ok=True)
if not SOURCE_DIR.is_dir() or not DERIVED_DIR.is_dir():
    raise FileNotFoundError(f"Authoritative v1.2 release bulunamadı: {DATASET_ROOT}")

with open(DERIVED_DIR / "dataset_metadata.json", encoding="utf-8") as file:
    dataset_metadata = json.load(file)
with open(DERIVED_DIR / "quality_report.json", encoding="utf-8") as file:
    quality_report = json.load(file)

dataset_info = dataset_metadata.get("dataset", {})
dataset_version = dataset_info.get("dataset_version")
generator_version = dataset_info.get("generator_version")
quality_gate = quality_report.get("report", {}).get("release_gate")
if dataset_version != DATASET_VERSION:
    raise RuntimeError(f"Dataset version mismatch: {dataset_version!r} != {DATASET_VERSION!r}")
if generator_version != DATASET_VERSION:
    raise RuntimeError(f"Generator version mismatch: {generator_version!r} != {DATASET_VERSION!r}")
if quality_gate != "PASS":
    raise RuntimeError(f"Dataset quality gate PASS değil: {quality_gate!r}")

snapshots = pd.read_parquet(DERIVED_DIR / "ml_maintenance_snapshots.parquet", engine="pyarrow")
next_service_targets = pd.read_parquet(DERIVED_DIR / "ml_next_service_targets.parquet", engine="pyarrow")
next_task_targets = pd.read_parquet(DERIVED_DIR / "ml_next_task_targets.parquet", engine="pyarrow")
split_manifest = pd.read_csv(DERIVED_DIR / "split_manifest.csv", encoding="utf-8-sig", low_memory=False)

# Referans tablolar schema ve taxonomy doğrulaması için okunur; feature üretmek için target kullanılmaz.
motorcycles = pd.read_csv(SOURCE_DIR / "motorcycles.csv", encoding="utf-8-sig", low_memory=False)
services = pd.read_csv(SOURCE_DIR / "services.csv", encoding="utf-8-sig", low_memory=False)
service_tasks = pd.read_csv(SOURCE_DIR / "service_tasks.csv", encoding="utf-8-sig", low_memory=False)
maintenance_tasks = pd.read_csv(SOURCE_DIR / "maintenance_tasks.csv", encoding="utf-8-sig", low_memory=False)
maintenance_policies = pd.read_csv(SOURCE_DIR / "maintenance_policies.csv", encoding="utf-8-sig", low_memory=False)
usage_profiles = pd.read_csv(SOURCE_DIR / "usage_profiles.csv", encoding="utf-8-sig", low_memory=False)

expected_shapes = {
    "snapshots": (41_518, 142),
    "next_service_targets": (41_518, 38),
    "next_task_targets": (41_518, 122),
    "split_manifest_rows": 41_518,
    "motorcycles_rows": 10_000,
    "services_rows": 41_518,
    "service_tasks_rows": 204_000,
}
actual_shapes = {
    "snapshots": snapshots.shape,
    "next_service_targets": next_service_targets.shape,
    "next_task_targets": next_task_targets.shape,
    "split_manifest_rows": len(split_manifest),
    "motorcycles_rows": len(motorcycles),
    "services_rows": len(services),
    "service_tasks_rows": len(service_tasks),
}
for name, expected in expected_shapes.items():
    if actual_shapes[name] != expected:
        raise RuntimeError(f"v1.2 structure guard failed: {name}={actual_shapes[name]} expected={expected}")

version_guard = pd.DataFrame([
    {"check": "dataset_version", "actual": dataset_version, "expected": DATASET_VERSION, "status": "PASS"},
    {"check": "generator_version", "actual": generator_version, "expected": DATASET_VERSION, "status": "PASS"},
    {"check": "quality_gate", "actual": quality_gate, "expected": "PASS", "status": "PASS"},
] + [{"check": name, "actual": value, "expected": expected_shapes[name], "status": "PASS"} for name, value in actual_shapes.items()])
display(version_guard)
print("DATASET_VERSION_GUARD=PASS")


,check,actual,expected,status
0,dataset_version,1.2.0,1.2.0,PASS
1,generator_version,1.2.0,1.2.0,PASS
2,quality_gate,PASS,PASS,PASS
3,snapshots,"(41518, 142)","(41518, 142)",PASS
4,next_service_targets,"(41518, 38)","(41518, 38)",PASS
5,next_task_targets,"(41518, 122)","(41518, 122)",PASS
6,split_manifest_rows,41518,41518,PASS
7,motorcycles_rows,10000,10000,PASS
8,services_rows,41518,41518,PASS
9,service_tasks_rows,204000,204000,PASS


DATASET_VERSION_GUARD=PASS


## 2. Feature table ile target table sınırı

### Ne yapıyoruz?
Ana girdiyi `ml_maintenance_snapshots.parquet` olarak belirliyoruz. Snapshot, belirli bir anda motosiklet hakkında bilinen bilgilerin fotoğrafıdır. Target tabloları ise bu fotoğraftan sonra gerçekleşen sonucu taşır.

### Neden yapıyoruz?
Feature ve target'ı ayrı tutmak, model girdisine gerçek cevabın yanlışlıkla karışmasını önler. Buna **leakage (bilgi sızıntısı)** denir: tahmin anında bilinmeyecek gelecekteki bilginin modele verilmesi.

### Yanlış yaparsak ne olur?
“Yarın yağmur yağacak mı?” modeline yarın ölçülen yağış miktarını vermek gibi, model çok başarılı görünür ama gerçek kullanımda cevap elinde olmaz. RideBase'te gerçek `next_service_date` bir feature olamaz.

### Bu dataset için kararımız
Snapshot kolonları feature adayı olarak incelenecek. Next-service ve next-task tabloları yalnız target contract, satır maskesi ve kalite kontrolünde kullanılacak; `X` feature matrisine girmeyecek.


In [2]:
table_inventory = pd.DataFrame([
    {"table": "ml_maintenance_snapshots", "role": "FEATURE_SOURCE", "rows": len(snapshots), "columns": snapshots.shape[1], "key": "snapshot_id"},
    {"table": "ml_next_service_targets", "role": "TARGET_ONLY", "rows": len(next_service_targets), "columns": next_service_targets.shape[1], "key": "snapshot_id"},
    {"table": "ml_next_task_targets", "role": "TARGET_ONLY", "rows": len(next_task_targets), "columns": next_task_targets.shape[1], "key": "snapshot_id"},
    {"table": "split_manifest", "role": "SPLIT_AND_ELIGIBILITY", "rows": len(split_manifest), "columns": split_manifest.shape[1], "key": "snapshot_id"},
])
display(table_inventory)
print("Feature source columns:", snapshots.columns.tolist())
print("Next-service target columns:", next_service_targets.columns.tolist())
print("Next-task label count:", sum(column.startswith("task__") for column in next_task_targets.columns))


,table,role,rows,columns,key
0,ml_maintenance_snapshots,FEATURE_SOURCE,41518,142,snapshot_id
1,ml_next_service_targets,TARGET_ONLY,41518,38,snapshot_id
2,ml_next_task_targets,TARGET_ONLY,41518,122,snapshot_id
3,split_manifest,SPLIT_AND_ELIGIBILITY,41518,37,snapshot_id


Feature source columns: ['snapshot_id', 'source_service_id', 'snapshot_at', 'snapshot_date', 'service_sequence', 'snapshot_year', 'snapshot_month', 'snapshot_quarter', 'snapshot_day_of_year', 'snapshot_month_sin', 'snapshot_month_cos', 'motorcycle_id', 'customer_id', 'workshop_id', 'model_id', 'brand', 'model_name', 'category', 'powertrain_type', 'engine_displacement_cc', 'cylinder_count', 'cooling_type', 'final_drive_type', 'transmission_type', 'engine_oil_service_qty_l', 'spark_plug_count', 'fuel_type', 'policy_group', 'policy_ready', 'spec_confidence', 'production_year', 'motorcycle_age_years', 'ownership_months', 'initial_mileage_km', 'is_left_truncated', 'days_observed', 'customer_type', 'is_fleet_customer', 'acquisition_channel', 'usage_type', 'annual_km_baseline', 'city_ratio', 'highway_ratio', 'offroad_ratio', 'track_ratio', 'avg_ride_days_per_week', 'daily_active_probability', 'avg_km_per_active_day', 'riding_intensity', 'seasonality_strength', 'winter_usage_factor', 'weather_

## 3. Dataset alignment ve güvenli join kontrolü

### Ne yapıyoruz?
Snapshot, iki target ve split tablosunun anahtarlarını karşılaştırıyoruz. Her tabloda `snapshot_id` tekil olmalı; hiçbir anahtar kayıp veya fazla olmamalı. Join sonrası satır sayısının değişmediğini açıkça raporluyoruz.

### Neden yapıyoruz?
Join, farklı tabloları ortak anahtarla birleştirir. Bir anahtar iki kez varsa tek satır çoğalabilir; bir anahtar yoksa satır kaybolabilir.

### Yanlış yaparsak ne olur?
Yanlış join, bir motosikletin feature'ını başka motosikletin sonucu ile eşleştirebilir. Bu, null doldurma hatasından daha tehlikelidir çünkü kod çalışır ve sahte ama makul görünen skor üretir.

### Bu dataset için kararımız
Bütün birleştirmeler `validate="one_to_one"` ile yapılacak. Uyuşmayan veya duplicate anahtar varsa notebook duracak.


In [3]:
def alignment_row(name, left, right, key="snapshot_id"):
    before = len(left)
    left_duplicate = int(left[key].duplicated().sum())
    right_duplicate = int(right[key].duplicated().sum())
    outer = left[[key]].merge(right[[key]], on=key, how="outer", indicator=True)
    checked = left[[key]].merge(right[[key]], on=key, how="left", validate="one_to_one", indicator=True)
    return {
        "join": name,
        "before_rows": before,
        "after_rows": len(checked),
        "matched": int(outer["_merge"].eq("both").sum()),
        "left_unmatched": int(outer["_merge"].eq("left_only").sum()),
        "right_unmatched": int(outer["_merge"].eq("right_only").sum()),
        "left_duplicates": left_duplicate,
        "right_duplicates": right_duplicate,
        "status": "PASS" if before == len(checked) and left_duplicate == right_duplicate == 0 and outer["_merge"].eq("both").all() else "FAIL",
    }


alignment_report = pd.DataFrame([
    alignment_row("snapshots ↔ next_service_targets", snapshots, next_service_targets),
    alignment_row("snapshots ↔ next_task_targets", snapshots, next_task_targets),
    alignment_row("snapshots ↔ split_manifest", snapshots, split_manifest),
])
display(alignment_report)
if alignment_report["status"].ne("PASS").any():
    raise RuntimeError("Snapshot/target/split alignment failed")
if not snapshots["motorcycle_id"].equals(next_service_targets["motorcycle_id"]):
    keyed = snapshots[["snapshot_id", "motorcycle_id"]].merge(next_service_targets[["snapshot_id", "motorcycle_id"]], on="snapshot_id", suffixes=("_feature", "_target"), validate="one_to_one")
    if not keyed["motorcycle_id_feature"].eq(keyed["motorcycle_id_target"]).all():
        raise RuntimeError("motorcycle_id feature/target alignment failed")


,join,before_rows,after_rows,matched,left_unmatched,right_unmatched,left_duplicates,right_duplicates,status
0,snapshots ↔ next_service_targets,41518,41518,41518,0,0,0,0,PASS
1,snapshots ↔ next_task_targets,41518,41518,41518,0,0,0,0,PASS
2,snapshots ↔ split_manifest,41518,41518,41518,0,0,0,0,PASS


## 4. TRAIN / VALIDATION / TEST ayrımı

### Ne yapıyoruz?
`split_manifest.csv` içindeki zaman bazlı ayrımı aynen kullanıyoruz.

- **TRAIN:** Preprocessing kurallarının ve gelecekte modelin öğrendiği bölüm.
- **VALIDATION:** Model seçimi ve ayar kararlarını kontrol edeceğimiz bölüm.
- **TEST:** En sonda tarafsız değerlendirme için saklanan bölüm.

**Fit**, bir kuralın veriden öğrenilmesidir; örneğin median veya kategoriler bulunur. **Transform**, önceden öğrenilmiş aynı kuralın yeni satırlara uygulanmasıdır.

### Neden yapıyoruz?
Servis verisi zamansaldır. Rastgele bölme gelecekteki servisi geçmişe karıştırabilir. Preprocessing bile validation/test üzerinden öğrenirse bu da leakage'dir.

### Yanlış yaparsak ne olur?
Yanlış: `median = tüm_dataset["annual_km"].median()`. Doğru: median yalnız TRAIN'den öğrenilir; validation ve test aynı median ile dönüştürülür.

### Bu dataset için kararımız
Random `train_test_split` kullanılmayacak. Splitler kesişmeyecek ve zaman sırası TRAIN → VALIDATION → TEST olarak doğrulanacak.


In [4]:
split_frame = split_manifest[["snapshot_id", "primary_time_split", "snapshot_at"]].copy()
split_frame["snapshot_at"] = pd.to_datetime(split_frame["snapshot_at"], errors="raise")
allowed_splits = {"TRAIN", "VALIDATION", "TEST"}
if set(split_frame["primary_time_split"].unique()) != allowed_splits:
    raise RuntimeError("Unexpected primary split labels")
if split_frame["snapshot_id"].duplicated().any():
    raise RuntimeError("Split manifest duplicate snapshot key")

split_summary = (
    split_frame.groupby("primary_time_split")
    .agg(rows=("snapshot_id", "size"), min_snapshot_at=("snapshot_at", "min"), max_snapshot_at=("snapshot_at", "max"))
    .reindex(["TRAIN", "VALIDATION", "TEST"])
    .reset_index()
)
temporal_order_pass = bool(
    split_summary.loc[0, "max_snapshot_at"] < split_summary.loc[1, "min_snapshot_at"]
    and split_summary.loc[1, "max_snapshot_at"] < split_summary.loc[2, "min_snapshot_at"]
)
if not temporal_order_pass:
    raise RuntimeError("Temporal split integrity failed")

split_by_snapshot = split_frame.set_index("snapshot_id")["primary_time_split"]
train_ids = split_frame.loc[split_frame["primary_time_split"].eq("TRAIN"), "snapshot_id"]
validation_ids = split_frame.loc[split_frame["primary_time_split"].eq("VALIDATION"), "snapshot_id"]
test_ids = split_frame.loc[split_frame["primary_time_split"].eq("TEST"), "snapshot_id"]
snapshot_indexed = snapshots.set_index("snapshot_id", drop=False)
train_snapshot = snapshot_indexed.loc[train_ids].copy()
validation_snapshot = snapshot_indexed.loc[validation_ids].copy()
test_snapshot = snapshot_indexed.loc[test_ids].copy()
display(split_summary)
print("SPLIT_INTEGRITY=PASS")


,primary_time_split,rows,min_snapshot_at,max_snapshot_at
0,TRAIN,27428,2021-01-27 11:31:00,2025-06-30 18:14:00
1,VALIDATION,6399,2025-07-01 08:34:00,2025-12-31 17:45:00
2,TEST,7691,2026-01-01 09:58:00,2026-08-03 17:33:00


SPLIT_INTEGRITY=PASS


## 5. Feature audit ve leakage audit

### Ne yapıyoruz?
142 snapshot kolonunun her birine görev veriyoruz: sayısal, kategorik, boolean, tarih, kimlik, metadata veya çıkarılacak kolon. **Cardinality**, bir kategorik kolondaki farklı değer sayısıdır. **Constant feature** hep aynı değeri; **near-constant feature** ise neredeyse hep aynı değeri taşır.

### Neden yapıyoruz?
Her sayısal kolon model girdisi değildir. `motorcycle_id` bir kimliktir; değeri büyüdükçe motosikletin “daha büyük” olduğu anlamına gelmez. Metadata ve geleceğe ait alanlar da çıkarılmalıdır.

### Yanlış yaparsak ne olur?
ID'leri ezberleyen model yeni motosiklette çalışmayabilir. Target/future kolonları modele girerse sahte başarı oluşur.

### Bu dataset için kararımız
Snapshot içindeki gerçek zamanlı ve geçmiş feature'lar adaydır. Snapshot/service/customer/motorcycle kimlikleri modelden çıkarılır ama modeling manifest'te izleme anahtarı olarak korunur. `workshop_id` (10) ve `model_id` (39) tahmin anında bilinen makul kategoriler olarak güvenli one-hot ile tutulur.


In [5]:
IDENTIFIER_COLUMNS = {"snapshot_id", "source_service_id", "motorcycle_id", "customer_id"}
RAW_DATETIME_COLUMNS = {"snapshot_at", "snapshot_date"}
METADATA_COLUMNS = {"feature_version", "data_origin", "generator_version", "random_seed", "scenario_id"}
REDUNDANT_DROP_COLUMNS = {"model_name", "workshop_city", "workshop_climate_zone", "fuel_type"}
BOOLEAN_COLUMNS = {
    "policy_ready", "is_left_truncated", "is_fleet_customer", "current_mileage_estimated",
    "current_is_breakdown", "current_is_warranty", "current_service_timestamp_repaired",
    "service_odometer_regression_count_to_date",
}
FUTURE_NAME_PATTERN = re.compile(r"(^|_)(next_service|next_task|future|target)(_|$)", re.I)

train_stats = train_snapshot.copy()
constant_columns = [column for column in snapshots.columns if train_stats[column].nunique(dropna=False) == 1]
near_constant_rows = []
for column in snapshots.columns:
    counts = train_stats[column].value_counts(dropna=False, normalize=True)
    dominant_rate = float(counts.iloc[0]) if len(counts) else np.nan
    if dominant_rate >= NEAR_CONSTANT_THRESHOLD:
        near_constant_rows.append({
            "column": column,
            "dominant_value": str(counts.index[0]),
            "dominant_rate": dominant_rate,
            "unique_count": int(train_stats[column].nunique(dropna=False)),
            "decision": "DROP" if column in constant_columns else "KEEP_AND_FLAG",
            "reason": "TRAIN'de sabit; bilgi taşımaz" if column in constant_columns else "Nadir fakat önemli olay olabilir; otomatik çıkarılmadı",
        })
near_constant_features = pd.DataFrame(near_constant_rows)


def is_boolean_column(column):
    return column in BOOLEAN_COLUMNS or pd.api.types.is_bool_dtype(snapshots[column])


feature_audit_rows = []
for column in snapshots.columns:
    series = snapshots[column]
    target_related = bool(FUTURE_NAME_PATTERN.search(column))
    future_information = target_related
    constant = column in constant_columns
    near_constant = bool(not near_constant_features.empty and column in set(near_constant_features["column"]))
    if column in IDENTIFIER_COLUMNS:
        role, action, reason = "IDENTIFIER", "DROP_FROM_MODEL_KEEP_IN_MANIFEST", "Kimlik/izleme anahtarı; genellenebilir sinyal değil"
    elif column in RAW_DATETIME_COLUMNS:
        role, action, reason = "DATETIME_FEATURE", "DROP_RAW_USE_DERIVED_TIME_FEATURES", "Raw timestamp yerine mevcut month/sin/cos/age feature'ları kullanılacak"
    elif column in METADATA_COLUMNS:
        role, action, reason = "METADATA", "DROP", "Dataset üretim metadata'sı; gerçek tahmin sinyali değil"
    elif future_information:
        role, action, reason = "FUTURE_LEAKAGE", "DROP", "Tahmin anında bilinmeyecek target/future bilgi"
    elif column in REDUNDANT_DROP_COLUMNS:
        role, action, reason = "DROP_CANDIDATE", "DROP_REDUNDANT", "Başka tutulan kolonla bire-bir aynı varlık bilgisini taşıyor"
    elif is_boolean_column(column):
        role, action, reason = "BOOLEAN_FEATURE", "KEEP", "Tahmin anında bilinen 0/1 durum"
    elif pd.api.types.is_numeric_dtype(series):
        role, action, reason = "NUMERIC_FEATURE", "KEEP", "Tahmin anında bilinen sayısal feature"
    else:
        role, action, reason = "CATEGORICAL_FEATURE", "KEEP_ONE_HOT", "Tahmin anında bilinen kategorik feature"
    if constant:
        action = "DROP_CONSTANT"
        reason = "TRAIN içinde tek değer; modele ayırıcı bilgi sağlamaz"
    candidate_feature = action in {"KEEP", "KEEP_ONE_HOT"}
    feature_audit_rows.append({
        "column_name": column,
        "dtype": str(series.dtype),
        "semantic_role": role,
        "candidate_feature": candidate_feature,
        "target_related": target_related,
        "future_information": future_information,
        "identifier": column in IDENTIFIER_COLUMNS,
        "constant": constant,
        "near_constant": near_constant,
        "null_rate": float(series.isna().mean()),
        "cardinality": int(series.nunique(dropna=True)),
        "recommended_action": action,
        "reason": reason,
    })
feature_audit = pd.DataFrame(feature_audit_rows)

leakage_rows = []
for column in snapshots.columns:
    future = bool(FUTURE_NAME_PATTERN.search(column))
    leakage_rows.append({
        "column": column, "source": "ml_maintenance_snapshots", "available_at_snapshot": not future,
        "future_information": future, "leakage_risk": "HIGH" if future else "NONE",
        "action": "DROP" if future else "AUDITED_CANDIDATE", "reason": "Future/target name and schema" if future else "Snapshot-time or historical field",
    })
for source_name, frame in [("ml_next_service_targets", next_service_targets), ("ml_next_task_targets", next_task_targets)]:
    for column in frame.columns:
        if column in {"snapshot_id", "source_service_id", "motorcycle_id", "customer_id", "workshop_id"}:
            continue
        leakage_rows.append({
            "column": column, "source": source_name, "available_at_snapshot": False,
            "future_information": True, "leakage_risk": "HIGH",
            "action": "TARGET/EVALUATION_ONLY", "reason": "Snapshot sonrasındaki sonuç veya censoring bilgisi; X'e giremez",
        })
leakage_audit = pd.DataFrame(leakage_rows)

feature_audit.to_csv(TABLES_DIR / "ml_feature_audit.csv", index=False, encoding="utf-8-sig")
leakage_audit.to_csv(TABLES_DIR / "ml_leakage_audit.csv", index=False, encoding="utf-8-sig")
display(feature_audit)
display(leakage_audit.query("leakage_risk == 'HIGH'").head(30))
print("Snapshot future/leakage columns:", int(feature_audit["future_information"].sum()))


,column_name,dtype,semantic_role,candidate_feature,target_related,future_information,identifier,constant,near_constant,null_rate,cardinality,recommended_action,reason
0,snapshot_id,object,IDENTIFIER,False,False,False,True,False,False,0.000000,41518,DROP_FROM_MODEL_KEEP_IN_MANIFEST,Kimlik/izleme anahtarı; genellenebilir sinyal değil
1,source_service_id,object,IDENTIFIER,False,False,False,True,False,False,0.000000,41518,DROP_FROM_MODEL_KEEP_IN_MANIFEST,Kimlik/izleme anahtarı; genellenebilir sinyal değil
2,snapshot_at,object,DATETIME_FEATURE,False,False,False,False,False,False,0.000000,39878,DROP_RAW_USE_DERIVED_TIME_FEATURES,Raw timestamp yerine mevcut month/sin/cos/age feature'ları kullanılacak
3,snapshot_date,object,DATETIME_FEATURE,False,False,False,False,False,False,0.000000,1674,DROP_RAW_USE_DERIVED_TIME_FEATURES,Raw timestamp yerine mevcut month/sin/cos/age feature'ları kullanılacak
4,service_sequence,int32,NUMERIC_FEATURE,True,False,False,False,False,False,0.000000,56,KEEP,Tahmin anında bilinen sayısal feature
5,snapshot_year,int32,NUMERIC_FEATURE,True,False,False,False,False,False,0.000000,6,KEEP,Tahmin anında bilinen sayısal feature
6,snapshot_month,int32,NUMERIC_FEATURE,True,False,False,False,False,False,0.000000,12,KEEP,Tahmin anında bilinen sayısal feature
7,snapshot_quarter,int32,NUMERIC_FEATURE,True,False,False,False,False,False,0.000000,4,KEEP,Tahmin anında bilinen sayısal feature
8,snapshot_day_of_year,int32,NUMERIC_FEATURE,True,False,False,False,False,False,0.000000,366,KEEP,Tahmin anında bilinen sayısal feature
9,snapshot_month_sin,float64,NUMERIC_FEATURE,True,False,False,False,False,False,0.000000,11,KEEP,Tahmin anında bilinen sayısal feature


,column,source,available_at_snapshot,future_information,leakage_risk,action,reason
142,snapshot_at,ml_next_service_targets,False,True,HIGH,TARGET/EVALUATION_ONLY,Snapshot sonrasındaki sonuç veya censoring bilgisi; X'e giremez
143,snapshot_odometer_km,ml_next_service_targets,False,True,HIGH,TARGET/EVALUATION_ONLY,Snapshot sonrasındaki sonuç veya censoring bilgisi; X'e giremez
144,target_event_observed,ml_next_service_targets,False,True,HIGH,TARGET/EVALUATION_ONLY,Snapshot sonrasındaki sonuç veya censoring bilgisi; X'e giremez
145,is_right_censored,ml_next_service_targets,False,True,HIGH,TARGET/EVALUATION_ONLY,Snapshot sonrasındaki sonuç veya censoring bilgisi; X'e giremez
146,next_service_id,ml_next_service_targets,False,True,HIGH,TARGET/EVALUATION_ONLY,Snapshot sonrasındaki sonuç veya censoring bilgisi; X'e giremez
147,next_service_received_at_raw,ml_next_service_targets,False,True,HIGH,TARGET/EVALUATION_ONLY,Snapshot sonrasındaki sonuç veya censoring bilgisi; X'e giremez
148,target_next_service_at,ml_next_service_targets,False,True,HIGH,TARGET/EVALUATION_ONLY,Snapshot sonrasındaki sonuç veya censoring bilgisi; X'e giremez
149,next_service_delivered_at,ml_next_service_targets,False,True,HIGH,TARGET/EVALUATION_ONLY,Snapshot sonrasındaki sonuç veya censoring bilgisi; X'e giremez
150,days_to_next_service_raw,ml_next_service_targets,False,True,HIGH,TARGET/EVALUATION_ONLY,Snapshot sonrasındaki sonuç veya censoring bilgisi; X'e giremez
151,days_to_next_service,ml_next_service_targets,False,True,HIGH,TARGET/EVALUATION_ONLY,Snapshot sonrasındaki sonuç veya censoring bilgisi; X'e giremez


Snapshot future/leakage columns: 0


## 6. NULL analizi ve semantik anlam

### Ne yapıyoruz?
Boş hücreleri yalnız saymıyor, ne anlama geldiklerini sınıflandırıyoruz. **Imputation**, gerçekten doldurulması gereken boş feature'a tanımlı bir yedek değer koymaktır.

- **TRUE MISSING:** Ölçülmesi gereken bilgi gerçekten eksik.
- **STRUCTURAL NULL:** Alan bu satır için uygulanamaz.
- **NOT YET OBSERVED:** Olay geçmişte henüz görülmemiş.
- **CENSORING-RELATED:** Sonuç kayıp değil; olay henüz gerçekleşmemiş.
- **UNKNOWN SOURCE VALUE:** Kaynak master değeri bilmiyor.

### Neden yapıyoruz?
NULL her zaman veri hatası değildir. Elektrikli motosiklette motor hacminin boş olması “ölçmeyi unuttuk” anlamına gelmez. Censored bir satırda next-service tarihinin boş olması da “servis henüz gözlenmedi” demektir.

### Yanlış yaparsak ne olur?
Censored `next_service_days` değerini 0 yapmak, motosiklet hemen servise gelmiş gibi yanlış target yaratır. Yapısal boşluğu median ile kapatmak, modelin “uygulanamaz” bilgisini kaybetmesine yol açabilir.

### Bu dataset için kararımız
Snapshot feature'larında yalnız 3 sayısal kolon boştur. Bunlar teknik master'ın uygulanmama/bilinmeme durumudur; `-1` ve ayrıca `*_was_missing` işaretiyle temsil edilecektir. Target NULL'larına dokunulmayacaktır.


In [6]:
null_semantics_rows = []
feature_null_columns = feature_audit.loc[feature_audit["candidate_feature"] & feature_audit["null_rate"].gt(0), "column_name"].tolist()
for column in feature_null_columns:
    if column in {"engine_displacement_cc", "engine_oil_service_qty_l", "spark_plug_count"}:
        null_type = "STRUCTURAL_NULL_OR_UNKNOWN_SOURCE_VALUE"
        strategy = "CONSTANT_-1_PLUS_MISSING_INDICATOR"
        reason = "EV için uygulanamaz olabilir veya teknik master değeri bilinmiyor; boşluk bilgisi korunmalı"
    else:
        null_type = "TRUE_MISSING"
        strategy = "TRAIN_MEDIAN_PLUS_MISSING_INDICATOR"
        reason = "Ölçülebilir numeric bilgi; robust TRAIN median düşünülebilir"
    null_semantics_rows.append({
        "column": column, "null_count": int(snapshots[column].isna().sum()),
        "null_rate": float(snapshots[column].isna().mean()), "null_type": null_type,
        "recommended_strategy": strategy, "reason": reason,
    })

target_null_examples = [
    ("ml_next_service_targets.days_to_next_service", int(next_service_targets["days_to_next_service"].isna().sum())),
    ("ml_next_service_targets.km_to_next_service", int(next_service_targets["km_to_next_service"].isna().sum())),
    ("ml_next_task_targets.task__*", int(next_task_targets[[c for c in next_task_targets if c.startswith("task__")]].isna().sum().sum())),
]
for column, count in target_null_examples:
    null_semantics_rows.append({
        "column": column, "null_count": count, "null_rate": np.nan,
        "null_type": "CENSORING_RELATED", "recommended_strategy": "PRESERVE_NULL_AND_EXCLUDE_WITH_MASK",
        "reason": "Olay/label henüz gözlenmedi; 0, -1, median, mode veya UNKNOWN yapılamaz",
    })
null_semantics = pd.DataFrame(null_semantics_rows)

task_label_columns = sorted(column for column in next_task_targets.columns if column.startswith("task__"))
censored_service = next_service_targets["is_right_censored"].eq(1)
censored_task = next_task_targets["is_right_censored"].eq(1)
target_censoring_checks = {
    "censored_next_service_values_nonnull": int(next_service_targets.loc[censored_service, ["days_to_next_service", "km_to_next_service", "target_next_service_at"]].notna().sum().sum()),
    "censored_task_label_values_nonnull": int(next_task_targets.loc[censored_task, task_label_columns].notna().sum().sum()),
    "observed_task_label_nulls": int(next_task_targets.loc[next_task_targets["target_event_observed"].eq(1), task_label_columns].isna().sum().sum()),
}
if target_censoring_checks["censored_next_service_values_nonnull"] != 0 or target_censoring_checks["censored_task_label_values_nonnull"] != 0:
    raise RuntimeError("Target censoring NULL semantics failed")

null_semantics.to_csv(TABLES_DIR / "ml_null_semantics.csv", index=False, encoding="utf-8-sig")
display(null_semantics)
display(pd.DataFrame([target_censoring_checks]))
print("TARGET_CENSORING_SEMANTICS=PASS")


,column,null_count,null_rate,null_type,recommended_strategy,reason
0,engine_displacement_cc,3187,0.076762,STRUCTURAL_NULL_OR_UNKNOWN_SOURCE_VALUE,CONSTANT_-1_PLUS_MISSING_INDICATOR,EV için uygulanamaz olabilir veya teknik master değeri bilinmiyor; boşluk bilgisi korunmalı
1,engine_oil_service_qty_l,36246,0.873019,STRUCTURAL_NULL_OR_UNKNOWN_SOURCE_VALUE,CONSTANT_-1_PLUS_MISSING_INDICATOR,EV için uygulanamaz olabilir veya teknik master değeri bilinmiyor; boşluk bilgisi korunmalı
2,spark_plug_count,663,0.015969,STRUCTURAL_NULL_OR_UNKNOWN_SOURCE_VALUE,CONSTANT_-1_PLUS_MISSING_INDICATOR,EV için uygulanamaz olabilir veya teknik master değeri bilinmiyor; boşluk bilgisi korunmalı
3,ml_next_service_targets.days_to_next_service,8442,NaN,CENSORING_RELATED,PRESERVE_NULL_AND_EXCLUDE_WITH_MASK,"Olay/label henüz gözlenmedi; 0, -1, median, mode veya UNKNOWN yapılamaz"
4,ml_next_service_targets.km_to_next_service,8442,NaN,CENSORING_RELATED,PRESERVE_NULL_AND_EXCLUDE_WITH_MASK,"Olay/label henüz gözlenmedi; 0, -1, median, mode veya UNKNOWN yapılamaz"
5,ml_next_task_targets.task__*,827316,NaN,CENSORING_RELATED,PRESERVE_NULL_AND_EXCLUDE_WITH_MASK,"Olay/label henüz gözlenmedi; 0, -1, median, mode veya UNKNOWN yapılamaz"


,censored_next_service_values_nonnull,censored_task_label_values_nonnull,observed_task_label_nulls
0,0,0,0


TARGET_CENSORING_SEMANTICS=PASS


## 7. Numeric feature analizi, imputation, missing indicator ve scaling

### Ne yapıyoruz?
Sayısal feature'ların dağılımını TRAIN üzerinde inceliyoruz. **Mean (ortalama)** toplamın satır sayısına bölünmesidir. **Median (ortanca)** sıralı değerlerin ortasındakidir. **Outlier**, çoğu gözlemden çok uzak değerdir; mean'i median'dan daha fazla çekebilir.

Eksik sayısal ölçümlerde median sık kullanılan sağlam bir seçenektir, fakat önce boşluğun anlamı kontrol edilmelidir. **Missing indicator**, doldurulan değerin aslında boş olduğunu ayrı 0/1 kolonuyla saklar.

**Scaling (ölçekleme)**, örneğin 5 yıllık yaş ile 80.000 km odometreyi benzer sayısal ölçekte sunar. `StandardScaler`, TRAIN ortalamasını yaklaşık 0 ve standart sapmasını yaklaşık 1 yapar. Linear/Cox türü modeller bundan faydalanır; tree modeller çoğu zaman ölçek istemez.

### Neden yapıyoruz?
Büyük sayılı kolonların linear modele yalnız ölçekleri nedeniyle baskın gelmesini ve boşluk anlamının kaybolmasını önlüyoruz.

### Yanlış yaparsak ne olur?
Median tüm dataset'ten hesaplanırsa test dağılımı TRAIN'e sızar. Outlier'ları otomatik silmek gerçek yüksek kilometre kullanımını yok edebilir.

### Bu dataset için kararımız
Numeric profil TRAIN'den hesaplanır; outlier otomatik silinmez. Üç teknik-master boşluğu `-1 + missing indicator` alır. Diğer numeric feature'larda boşluk yoktur. Bütün numeric feature'lar linear/survival preprocessor'da TRAIN ile ölçeklenir.


In [7]:
candidate_columns_pre_constant = feature_audit.loc[
    ~feature_audit["identifier"]
    & ~feature_audit["future_information"]
    & ~feature_audit["semantic_role"].isin(["DATETIME_FEATURE", "METADATA", "DROP_CANDIDATE"]),
    "column_name",
].tolist()
candidate_columns = feature_audit.loc[feature_audit["candidate_feature"], "column_name"].tolist()
numeric_candidates = [column for column in candidate_columns if feature_audit.set_index("column_name").loc[column, "semantic_role"] == "NUMERIC_FEATURE"]
boolean_features = [column for column in candidate_columns if feature_audit.set_index("column_name").loc[column, "semantic_role"] == "BOOLEAN_FEATURE"]
categorical_candidates = [column for column in candidate_columns if feature_audit.set_index("column_name").loc[column, "semantic_role"] == "CATEGORICAL_FEATURE"]

numeric_profile_rows = []
for column in numeric_candidates:
    values = pd.to_numeric(train_snapshot[column], errors="coerce")
    numeric_profile_rows.append({
        "column": column, "dtype": str(values.dtype), "min": values.min(), "max": values.max(),
        "mean": values.mean(), "median": values.median(), "std": values.std(),
        "p01": values.quantile(.01), "p25": values.quantile(.25), "p50": values.quantile(.50),
        "p75": values.quantile(.75), "p99": values.quantile(.99),
        "null_rate": values.isna().mean(), "unique_count": values.nunique(dropna=True),
        "outlier_candidate": bool(
            pd.notna(values.quantile(.99)) and pd.notna(values.max())
            and abs(values.max()) > max(abs(values.quantile(.99)) * 5, 10)
        ),
        "decision": "KEEP; NO AUTOMATIC ROW REMOVAL",
    })
numeric_profile = pd.DataFrame(numeric_profile_rows)

structural_numeric_features = [column for column in numeric_candidates if column in feature_null_columns]
complete_numeric_features = [column for column in numeric_candidates if column not in structural_numeric_features]
median_numeric_features = []  # v1.2'de TRUE_MISSING numeric candidate bulunmadı.
missing_indicator_features = structural_numeric_features.copy()

numeric_profile.to_csv(TABLES_DIR / "ml_numeric_feature_profile.csv", index=False, encoding="utf-8-sig")
display(numeric_profile)
print("Complete numeric:", len(complete_numeric_features))
print("Structural/unknown numeric with -1 + indicator:", structural_numeric_features)
print("TRAIN-median numeric in current v1.2:", median_numeric_features)


,column,dtype,min,max,mean,median,std,p01,p25,p50,p75,p99,null_rate,unique_count,outlier_candidate,decision
0,service_sequence,int32,1.000000,43.000000,4.826200,3.000000e+00,5.143907,1.000000,2.000000,3.000000e+00,6.000000,26.000000,0.000000,43,False,KEEP; NO AUTOMATIC ROW REMOVAL
1,snapshot_year,int32,2021.000000,2025.000000,2023.510719,2.024000e+03,1.098228,2021.000000,2023.000000,2.024000e+03,2024.000000,2025.000000,0.000000,5,False,KEEP; NO AUTOMATIC ROW REMOVAL
2,snapshot_month,int32,1.000000,12.000000,6.456723,6.000000e+00,3.263512,1.000000,4.000000,6.000000e+00,9.000000,12.000000,0.000000,12,False,KEEP; NO AUTOMATIC ROW REMOVAL
3,snapshot_quarter,int32,1.000000,4.000000,2.477614,2.000000e+00,1.073098,1.000000,2.000000,2.000000e+00,3.000000,4.000000,0.000000,4,False,KEEP; NO AUTOMATIC ROW REMOVAL
4,snapshot_day_of_year,int32,1.000000,366.000000,181.293642,1.725000e+02,99.755614,5.000000,100.000000,1.725000e+02,267.000000,360.000000,0.000000,366,False,KEEP; NO AUTOMATIC ROW REMOVAL
5,snapshot_month_sin,float64,-1.000000,1.000000,0.059571,1.224647e-16,0.725461,-1.000000,-0.500000,1.224647e-16,0.866025,1.000000,0.000000,11,False,KEEP; NO AUTOMATIC ROW REMOVAL
6,snapshot_month_cos,float64,-1.000000,1.000000,-0.067085,-1.836970e-16,0.682417,-1.000000,-0.866025,-1.836970e-16,0.500000,1.000000,0.000000,11,False,KEEP; NO AUTOMATIC ROW REMOVAL
7,engine_displacement_cc,float64,49.000000,649.000000,105.750324,1.240000e+02,69.444272,49.000000,50.000000,1.240000e+02,125.000000,349.000000,0.075397,18,False,KEEP; NO AUTOMATIC ROW REMOVAL
8,cylinder_count,int64,0.000000,2.000000,1.004594,1.000000e+00,0.081778,1.000000,1.000000,1.000000e+00,1.000000,1.000000,0.000000,3,False,KEEP; NO AUTOMATIC ROW REMOVAL
9,engine_oil_service_qty_l,float64,0.000000,1.500000,0.893590,8.000000e-01,0.243386,0.800000,0.800000,8.000000e-01,0.800000,1.500000,0.874289,4,False,KEEP; NO AUTOMATIC ROW REMOVAL


Complete numeric: 92
Structural/unknown numeric with -1 + indicator: ['engine_displacement_cc', 'engine_oil_service_qty_l', 'spark_plug_count']
TRAIN-median numeric in current v1.2: []


## 8. Categorical analiz, cardinality, UNKNOWN ve one-hot encoding

### Ne yapıyoruz?
Kategorik kolonların farklı değer sayılarını ve nadir kategorilerini TRAIN üzerinde ölçüyoruz. **One-hot encoding**, `COMMUTER/COURIER/TRACK` gibi değerleri ayrı 0/1 kolonlarına çevirir.

`COMMUTER=1, COURIER=2, TRACK=3` yapmak yanlıştır; model 3'ün 1'den “daha büyük” olduğunu sanabilir. One-hot bu sahte sıralamayı oluşturmaz.

**UNKNOWN**, değer bilinmiyor demektir. **NOT_APPLICABLE**, özelliğin o varlık için geçerli olmaması demektir. Bunlar aynı değildir.

### Neden yapıyoruz?
Model metin kategorilerini doğrudan hesaplayamaz. Validation/test'te TRAIN'de görülmeyen kategori olabilir; `handle_unknown="ignore"` bu yeni kategoride transform'un çökmesini önler.

### Yanlış yaparsak ne olur?
8.442 farklı `motorcycle_id` one-hot yapılırsa model motosikletleri ezberler, matris gereksiz büyür ve yeni motosiklette genelleme zayıflar. TRAIN dışı kategoriler encoder fit'ine katılırsa leakage oluşur.

### Bu dataset için kararımız
Kimlikler çıkarılır. `workshop_id`, `model_id`, brand ve diğer makul cardinality kategorileri one-hot edilir. Kategorik boşluk şu an yoktur; reusable pipeline gelecekteki gerçek missing için `UNKNOWN` kullanır. Encoder kategorileri yalnız TRAIN'den öğrenir.


In [8]:
def cardinality_class(value):
    if value <= 10:
        return "LOW"
    if value <= 50:
        return "MEDIUM"
    if value <= 200:
        return "HIGH"
    return "VERY_HIGH"


categorical_profile_rows = []
for column in categorical_candidates:
    counts = train_snapshot[column].astype("string").value_counts(dropna=False)
    unique_count = int(train_snapshot[column].nunique(dropna=True))
    categorical_profile_rows.append({
        "column": column, "unique_count": unique_count,
        "null_count": int(train_snapshot[column].isna().sum()),
        "null_rate": float(train_snapshot[column].isna().mean()),
        "top_categories": " | ".join(f"{index}:{value}" for index, value in counts.head(5).items()),
        "rare_category_count": int((counts < RARE_CATEGORY_MIN_COUNT).sum()),
        "cardinality_class": cardinality_class(unique_count),
    })
categorical_profile = pd.DataFrame(categorical_profile_rows).sort_values(["unique_count", "column"], ascending=[False, True])

high_cardinality_decisions = pd.DataFrame([
    {"column": "snapshot_id", "cardinality": snapshots["snapshot_id"].nunique(), "use_or_drop": "DROP_FROM_X_KEEP_MANIFEST", "encoding_strategy": "NONE", "reason": "Satır kimliği"},
    {"column": "source_service_id", "cardinality": snapshots["source_service_id"].nunique(), "use_or_drop": "DROP_FROM_X", "encoding_strategy": "NONE", "reason": "Servis kaydı kimliği"},
    {"column": "motorcycle_id", "cardinality": snapshots["motorcycle_id"].nunique(), "use_or_drop": "DROP_FROM_X_KEEP_MANIFEST", "encoding_strategy": "NONE", "reason": "Yüksek cardinality kimliği; ezberleme riski"},
    {"column": "customer_id", "cardinality": snapshots["customer_id"].nunique(), "use_or_drop": "DROP_FROM_X", "encoding_strategy": "NONE", "reason": "Yüksek cardinality kişi kimliği"},
    {"column": "workshop_id", "cardinality": snapshots["workshop_id"].nunique(), "use_or_drop": "KEEP", "encoding_strategy": "ONE_HOT_HANDLE_UNKNOWN", "reason": "10 kategori; prediction anında biliniyor, unseen workshop güvenli"},
    {"column": "model_id", "cardinality": snapshots["model_id"].nunique(), "use_or_drop": "KEEP", "encoding_strategy": "ONE_HOT_HANDLE_UNKNOWN", "reason": "39 model; makul cardinality ve bakım davranışıyla ilişkili"},
    {"column": "brand", "cardinality": snapshots["brand"].nunique(), "use_or_drop": "KEEP", "encoding_strategy": "ONE_HOT_HANDLE_UNKNOWN", "reason": "Düşük/orta cardinality"},
    {"column": "model_name", "cardinality": snapshots["model_name"].nunique(), "use_or_drop": "DROP_REDUNDANT", "encoding_strategy": "NONE", "reason": "model_id ile bire-bir aynı varlığı temsil ediyor"},
])

categorical_profile.to_csv(TABLES_DIR / "ml_categorical_feature_profile.csv", index=False, encoding="utf-8-sig")
high_cardinality_decisions.to_csv(TABLES_DIR / "ml_high_cardinality_decisions.csv", index=False, encoding="utf-8-sig")
display(categorical_profile)
display(high_cardinality_decisions)


,column,unique_count,null_count,null_rate,top_categories,rare_category_count,cardinality_class
1,model_id,39,0,0.0,KUBA_BLUEBERRY50:3018 | MONDIAL_WING50I:2367 | HONDA_ACTIVA125:2201 | HONDA_PCX125:2200 | MONDIAL_SFC_MINI_50EC:2177,6,MEDIUM
8,policy_group,20,0,0.0,ICE_50_AIR_BELT:5777 | ICE_50_AIR_CHAIN:5570 | ICE_125_LIQUID_BELT:4927 | ICE_125_SCOOTER_BELT:2201 | ICE_125_AIR_CHAIN:2141,4,MEDIUM
2,brand,12,0,0.0,Honda:7979 | Mondial:6775 | Kuba:4572 | Yamaha:2455 | TVS:2120,1,MEDIUM
3,category,12,0,0.0,SCOOTER:16674 | CUB:6844 | NAKED:2238 | BIG_SCOOTER:1141 | SCRAMBLER:105,3,MEDIUM
0,workshop_id,10,0,0.0,WS0007:3382 | WS0008:3370 | WS0001:3188 | WS0003:2952 | WS0005:2809,0,LOW
11,acquisition_channel,7,0,0.0,REFERRAL:7275 | ORGANIC:6399 | GOOGLE_MAPS:3249 | DIRECT_SALES:3233 | WALK_IN:3192,0,LOW
15,climate_zone,7,0,0.0,MEDITERRANEAN_HOT:6191 | TEMPERATE_HUMID:5433 | CONTINENTAL_DRY:5080 | SEMI_ARID_HOT:3370 | MEDITERRANEAN_COASTAL:2952,0,LOW
7,transmission_type,7,0,0.0,CVT:15056 | MANUAL_5:4687 | AUTOMATIC:2759 | SEMI_AUTO_4:2177 | MANUAL_6:1446,1,LOW
12,usage_type,7,0,0.0,COMMUTER:12809 | COURIER:5259 | WEEKEND:3993 | SEASONAL:2781 | TOURING:1661,0,LOW
16,workshop_region,6,0,0.0,İç Anadolu:7166 | Akdeniz:6191 | Marmara:5433 | Güneydoğu Anadolu:3370 | Ege:2952,0,LOW


,column,cardinality,use_or_drop,encoding_strategy,reason
0,snapshot_id,41518,DROP_FROM_X_KEEP_MANIFEST,NONE,Satır kimliği
1,source_service_id,41518,DROP_FROM_X,NONE,Servis kaydı kimliği
2,motorcycle_id,8442,DROP_FROM_X_KEEP_MANIFEST,NONE,Yüksek cardinality kimliği; ezberleme riski
3,customer_id,6080,DROP_FROM_X,NONE,Yüksek cardinality kişi kimliği
4,workshop_id,10,KEEP,ONE_HOT_HANDLE_UNKNOWN,"10 kategori; prediction anında biliniyor, unseen workshop güvenli"
5,model_id,39,KEEP,ONE_HOT_HANDLE_UNKNOWN,39 model; makul cardinality ve bakım davranışıyla ilişkili
6,brand,12,KEEP,ONE_HOT_HANDLE_UNKNOWN,Düşük/orta cardinality
7,model_name,39,DROP_REDUNDANT,NONE,model_id ile bire-bir aynı varlığı temsil ediyor


## 9. Constant, near-constant, redundant, datetime ve boolean kararları

### Ne yapıyoruz?
TRAIN'de hep aynı kalan kolonları çıkarıyor, ≥%99 aynı değere sahip kolonları işaretliyor ve aynı bilgiyi başka adla tekrar eden kolonları inceliyoruz. Boolean feature yalnız iki durumu (0/1) temsil eder.

### Neden yapıyoruz?
Constant kolon ayırıcı bilgi sağlamaz. Redundant kolonlar gereksiz boyut ve bazı modellerde katsayı kararsızlığı yaratabilir. Raw datetime'ı epoch sayısına çevirmek anlamsız uzaklık ve veri dönemi ezberleme riski yaratır.

### Yanlış yaparsak ne olur?
Near-constant bir breakdown/warranty flag'ini otomatik silmek nadir ama önemli sinyali yok edebilir. Yüksek correlation da tek başına çıkarma sebebi değildir; iki feature aynı bilgiyi farklı biçimde taşıyabilir.

### Bu dataset için kararımız
Beş üretim metadata kolonu constant olduğu için çıkarılır. Near-constant ama operasyonel anlamlı flag'ler korunur. Raw `snapshot_at/date` çıkarılır; mevcut month, season/sin/cos ve vehicle-age feature'ları tutulur. Model adı, workshop şehri, duplicate workshop climate ve fuel type redundant olarak çıkarılır.


In [9]:
redundancy_rows = []
for left, right, decision in [
    ("model_id", "model_name", "KEEP model_id; DROP model_name"),
    ("workshop_id", "workshop_city", "KEEP workshop_id; DROP workshop_city"),
    ("climate_zone", "workshop_climate_zone", "KEEP climate_zone; DROP workshop_climate_zone"),
    ("powertrain_type", "fuel_type", "KEEP powertrain_type; DROP fuel_type"),
]:
    left_to_right = int(train_snapshot.groupby(left, dropna=False)[right].nunique(dropna=False).max())
    right_to_left = int(train_snapshot.groupby(right, dropna=False)[left].nunique(dropna=False).max())
    redundancy_rows.append({
        "feature_a": left, "feature_b": right,
        "a_to_b_max_unique": left_to_right, "b_to_a_max_unique": right_to_left,
        "relationship": "ONE_TO_ONE" if left_to_right == right_to_left == 1 else "NOT_EXACT",
        "decision": decision,
        "reason": "Aynı varlığın iki etiketi; başlangıç pipeline'ında tek temsil yeterli" if left_to_right == right_to_left == 1 else "Sadece raporla; correlation tek başına drop değildir",
    })
redundancy_report = pd.DataFrame(redundancy_rows)

final_raw_features = complete_numeric_features + structural_numeric_features + boolean_features + categorical_candidates
final_raw_features = [column for column in final_raw_features if column not in constant_columns and column not in REDUNDANT_DROP_COLUMNS]
complete_numeric_features = [column for column in complete_numeric_features if column in final_raw_features]
structural_numeric_features = [column for column in structural_numeric_features if column in final_raw_features]
boolean_features = [column for column in boolean_features if column in final_raw_features]
categorical_features = [column for column in categorical_candidates if column in final_raw_features]

if set(final_raw_features) & (IDENTIFIER_COLUMNS | RAW_DATETIME_COLUMNS | METADATA_COLUMNS | REDUNDANT_DROP_COLUMNS):
    raise RuntimeError("Dropped identifier/datetime/metadata/redundant column leaked into final raw features")
if len(final_raw_features) != len(set(final_raw_features)):
    raise RuntimeError("Duplicate final raw feature")

near_constant_features.to_csv(TABLES_DIR / "ml_constant_near_constant_features.csv", index=False, encoding="utf-8-sig")
redundancy_report.to_csv(TABLES_DIR / "ml_redundant_feature_audit.csv", index=False, encoding="utf-8-sig")
display(near_constant_features)
display(redundancy_report)
print("Constant removed:", constant_columns)
print("Final raw feature count:", len(final_raw_features))
print("Numeric / categorical / boolean:", len(complete_numeric_features) + len(structural_numeric_features), len(categorical_features), len(boolean_features))


,column,dominant_value,dominant_rate,unique_count,decision,reason
0,powertrain_type,ICE,0.998943,2,KEEP_AND_FLAG,Nadir fakat önemli olay olabilir; otomatik çıkarılmadı
1,cylinder_count,1,0.993292,3,KEEP_AND_FLAG,Nadir fakat önemli olay olabilir; otomatik çıkarılmadı
2,fuel_type,GASOLINE,0.998943,2,KEEP_AND_FLAG,Nadir fakat önemli olay olabilir; otomatik çıkarılmadı
3,is_left_truncated,1,0.995698,2,KEEP_AND_FLAG,Nadir fakat önemli olay olabilir; otomatik çıkarılmadı
4,current_is_breakdown,0,0.990156,2,KEEP_AND_FLAG,Nadir fakat önemli olay olabilir; otomatik çıkarılmadı
5,current_is_warranty,0,0.993693,2,KEEP_AND_FLAG,Nadir fakat önemli olay olabilir; otomatik çıkarılmadı
6,service_odometer_regression_count_to_date,0,0.999854,2,KEEP_AND_FLAG,Nadir fakat önemli olay olabilir; otomatik çıkarılmadı
7,feature_version,1.2.0,1.000000,1,DROP,TRAIN'de sabit; bilgi taşımaz
8,data_origin,SYNTHETIC,1.000000,1,DROP,TRAIN'de sabit; bilgi taşımaz
9,generator_version,1.2.0,1.000000,1,DROP,TRAIN'de sabit; bilgi taşımaz


,feature_a,feature_b,a_to_b_max_unique,b_to_a_max_unique,relationship,decision,reason
0,model_id,model_name,1,1,ONE_TO_ONE,KEEP model_id; DROP model_name,Aynı varlığın iki etiketi; başlangıç pipeline'ında tek temsil yeterli
1,workshop_id,workshop_city,1,1,ONE_TO_ONE,KEEP workshop_id; DROP workshop_city,Aynı varlığın iki etiketi; başlangıç pipeline'ında tek temsil yeterli
2,climate_zone,workshop_climate_zone,1,1,ONE_TO_ONE,KEEP climate_zone; DROP workshop_climate_zone,Aynı varlığın iki etiketi; başlangıç pipeline'ında tek temsil yeterli
3,powertrain_type,fuel_type,1,1,ONE_TO_ONE,KEEP powertrain_type; DROP fuel_type,Aynı varlığın iki etiketi; başlangıç pipeline'ında tek temsil yeterli


Constant removed: ['feature_version', 'data_origin', 'generator_version', 'random_seed', 'scenario_id']
Final raw feature count: 127
Numeric / categorical / boolean: 95 24 8


## 10. Reusable ColumnTransformer pipeline

### Ne yapıyoruz?
Bir **pipeline**, işlemleri sabit sırada bir zincir haline getirir. `ColumnTransformer`, farklı kolon gruplarına farklı zincir uygular:

- Tam numeric: `StandardScaler`
- Yapısal/unknown numeric: `SimpleImputer(-1, add_indicator=True) → StandardScaler`
- Boolean: eksik olursa en sık TRAIN değeri, sonra 0/1
- Kategorik: `SimpleImputer("UNKNOWN") → OneHotEncoder(handle_unknown="ignore")`

**Sparse matrix**, çoğu hücresi 0 olan matrisi yalnız dolu değerleri saklayarak temsil eder. One-hot sonrası yüzlerce 0 oluştuğu için dense `.toarray()` gereksiz RAM tüketir.

### Neden yapıyoruz?
Aynı fitted nesnenin sonraki V1/V2/V3 notebooklarında tekrar kullanılması, kolon sırasının ve kuralların değişmesini önler.

### Yanlış yaparsak ne olur?
Validation/test'e ayrı imputer veya encoder fit etmek, her splitte farklı kolon ve istatistik üretir. Validation/Test'ten preprocessing bile öğrenmek leakage'dir.

### Bu dataset için kararımız
`LINEAR_SURVIVAL_PREPROCESSOR` yalnız TRAIN üzerinde fit edilecek. Tree model ileride scaling istemezse aynı feature kararlarıyla ayrı config kullanabilir; bu notebook yalnız linear/survival-ready scaled artifact üretir.


In [10]:
complete_numeric_pipeline = Pipeline([
    ("scaler", StandardScaler()),
])
structural_numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="constant", fill_value=-1.0, add_indicator=True)),
    ("scaler", StandardScaler()),
])
boolean_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
])
categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="constant", fill_value="UNKNOWN")),
    ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=True, dtype=np.float32)),
])

transformers = []
if complete_numeric_features:
    transformers.append(("numeric", complete_numeric_pipeline, complete_numeric_features))
if structural_numeric_features:
    transformers.append(("numeric_structural", structural_numeric_pipeline, structural_numeric_features))
if boolean_features:
    transformers.append(("boolean", boolean_pipeline, boolean_features))
if categorical_features:
    transformers.append(("categorical", categorical_pipeline, categorical_features))

LINEAR_SURVIVAL_PREPROCESSOR = ColumnTransformer(
    transformers=transformers,
    remainder="drop",
    sparse_threshold=1.0,
    verbose_feature_names_out=True,
)

X_raw = snapshots.set_index("snapshot_id")[final_raw_features].copy()
X_train = X_raw.loc[train_ids]
X_validation = X_raw.loc[validation_ids]
X_test = X_raw.loc[test_ids]

LINEAR_SURVIVAL_PREPROCESSOR.fit(X_train)
X_train_processed = LINEAR_SURVIVAL_PREPROCESSOR.transform(X_train)
X_validation_processed = LINEAR_SURVIVAL_PREPROCESSOR.transform(X_validation)
X_test_processed = LINEAR_SURVIVAL_PREPROCESSOR.transform(X_test)

preprocessor_path = MODELS_DIR / "ml_preprocessor_v1_2.joblib"
joblib.dump(LINEAR_SURVIVAL_PREPROCESSOR, preprocessor_path)
reloaded_preprocessor = joblib.load(preprocessor_path)
sample_before = LINEAR_SURVIVAL_PREPROCESSOR.transform(X_test.iloc[:100])
sample_after = reloaded_preprocessor.transform(X_test.iloc[:100])
reload_difference = sample_before - sample_after
reload_pass = reload_difference.nnz == 0 if sparse.issparse(reload_difference) else bool(np.allclose(reload_difference, 0))
if not reload_pass:
    raise RuntimeError("Saved preprocessor reload transform mismatch")

print("Processed shapes:", X_train_processed.shape, X_validation_processed.shape, X_test_processed.shape)
print("Sparse outputs:", sparse.issparse(X_train_processed), sparse.issparse(X_validation_processed), sparse.issparse(X_test_processed))
print("PREPROCESSOR_RELOAD=PASS")


Processed shapes: (27428, 277) (6399, 277) (7691, 277)
Sparse outputs: True True True
PREPROCESSOR_RELOAD=PASS


## 11. Transform sonrası kalite, feature isimleri ve imputation raporu

### Ne yapıyoruz?
Train/validation/test satır sayısının korunduğunu, üç matrisin aynı kolon sayısına sahip olduğunu ve feature matrisinde NaN/sonsuz değer kalmadığını kontrol ediyoruz. One-hot sonrası üretilen kolon adlarını çıkarıyoruz.

### Neden yapıyoruz?
Model yalnız aynı feature düzenindeki matrisleri kabul eder. One-hot sonrası kolon sayısı artar; çünkü her kategori ayrı 0/1 kolona dönüşür.

### Yanlış yaparsak ne olur?
Bir splitte farklı kategori kolonları oluşursa katsayılar yanlış feature'a uygulanabilir. Dense dönüşüm gereksiz bellek tüketebilir.

### Bu dataset için kararımız
Bütün matrisler sparse kalacak. Feature matrix NaN=0 olmalı; target NULL'ları bu kontrolden ayrıdır ve korunacaktır.

### Sonuç nasıl yorumlanmalı?
Raw feature sayısından encoded boyuta artış hata değildir; kategorilerin güvenli one-hot temsilleridir.


In [11]:
def matrix_quality(name, matrix, expected_rows):
    values = matrix.data if sparse.issparse(matrix) else np.asarray(matrix)
    return {
        "split": name,
        "row_count": matrix.shape[0],
        "expected_rows": expected_rows,
        "column_count": matrix.shape[1],
        "nan_count": int(np.isnan(values).sum()),
        "inf_count": int(np.isinf(values).sum()),
        "sparse": bool(sparse.issparse(matrix)),
        "status": "PASS" if matrix.shape[0] == expected_rows and np.isnan(values).sum() == 0 and np.isinf(values).sum() == 0 else "FAIL",
    }


transformed_quality = pd.DataFrame([
    matrix_quality("TRAIN", X_train_processed, len(train_ids)),
    matrix_quality("VALIDATION", X_validation_processed, len(validation_ids)),
    matrix_quality("TEST", X_test_processed, len(test_ids)),
])
if transformed_quality["column_count"].nunique() != 1:
    transformed_quality["status"] = "FAIL"
if transformed_quality["status"].ne("PASS").any():
    raise RuntimeError("Transformed feature quality failed")

encoded_feature_names = LINEAR_SURVIVAL_PREPROCESSOR.get_feature_names_out().tolist()


def source_from_encoded(name):
    transformer, rest = name.split("__", 1)
    if transformer in {"numeric", "boolean"}:
        return rest
    if transformer == "numeric_structural":
        if rest.startswith("missingindicator_"):
            return rest.removeprefix("missingindicator_")
        return rest
    if transformer == "categorical":
        matches = [column for column in categorical_features if rest == column or rest.startswith(column + "_")]
        return max(matches, key=len) if matches else rest
    return rest


model_feature_rows = []
for name in encoded_feature_names:
    source_column = source_from_encoded(name)
    if name.startswith("categorical__"):
        feature_type, transformation = "ONE_HOT", "TRAIN categories; handle_unknown=ignore"
    elif "missingindicator_" in name:
        feature_type, transformation = "MISSING_INDICATOR", "1 if original numeric value was NULL"
    elif name.startswith("boolean__"):
        feature_type, transformation = "BOOLEAN", "most_frequent fallback; 0/1"
    else:
        feature_type, transformation = "NUMERIC", "StandardScaler fitted on TRAIN"
    model_feature_rows.append({
        "feature_name": name, "source_column": source_column, "feature_type": feature_type,
        "transformation": transformation, "model_ready": True,
    })
model_feature_list = pd.DataFrame(model_feature_rows)

imputation_rows = []
for column in final_raw_features:
    if column in structural_numeric_features:
        feature_type, strategy, fill = "NUMERIC", "CONSTANT_-1_PLUS_MISSING_INDICATOR", -1.0
        reason = "Structural/unknown technical master NULL; missing meaning retained"
    elif column in categorical_features:
        feature_type, strategy, fill = "CATEGORICAL", "CONSTANT_UNKNOWN_THEN_ONE_HOT", "UNKNOWN"
        reason = "Future gerçek missing kategori için explicit unknown; v1.2'de categorical NULL yok"
    elif column in boolean_features:
        feature_type, strategy, fill = "BOOLEAN", "MOST_FREQUENT_IF_NEEDED", "TRAIN_MODE"
        reason = "Boolean NULL oluşursa yalnız TRAIN mode; mevcut v1.2'de NULL yok"
    else:
        feature_type, strategy, fill = "NUMERIC", "NO_IMPUTATION_NEEDED", pd.NA
        reason = "v1.2 candidate feature complete"
    imputation_rows.append({
        "column": column, "feature_type": feature_type,
        "train_null_before": int(X_train[column].isna().sum()),
        "val_null_before": int(X_validation[column].isna().sum()),
        "test_null_before": int(X_test[column].isna().sum()),
        "strategy": strategy, "fit_source": "TRAIN_ONLY", "fill_value_if_applicable": fill,
        "train_null_after": 0, "val_null_after": 0, "test_null_after": 0, "reason": reason,
    })
imputation_report = pd.DataFrame(imputation_rows)

model_feature_list.to_csv(TABLES_DIR / "model_feature_list.csv", index=False, encoding="utf-8-sig")
imputation_report.to_csv(TABLES_DIR / "ml_imputation_report.csv", index=False, encoding="utf-8-sig")
transformed_quality.to_csv(TABLES_DIR / "ml_transformed_data_quality.csv", index=False, encoding="utf-8-sig")
display(transformed_quality)
display(model_feature_list.head(40))
display(imputation_report.loc[imputation_report[["train_null_before", "val_null_before", "test_null_before"]].sum(axis=1).gt(0)])


,split,row_count,expected_rows,column_count,nan_count,inf_count,sparse,status
0,TRAIN,27428,27428,277,0,0,True,PASS
1,VALIDATION,6399,6399,277,0,0,True,PASS
2,TEST,7691,7691,277,0,0,True,PASS


,feature_name,source_column,feature_type,transformation,model_ready
0,numeric__service_sequence,service_sequence,NUMERIC,StandardScaler fitted on TRAIN,True
1,numeric__snapshot_year,snapshot_year,NUMERIC,StandardScaler fitted on TRAIN,True
2,numeric__snapshot_month,snapshot_month,NUMERIC,StandardScaler fitted on TRAIN,True
3,numeric__snapshot_quarter,snapshot_quarter,NUMERIC,StandardScaler fitted on TRAIN,True
4,numeric__snapshot_day_of_year,snapshot_day_of_year,NUMERIC,StandardScaler fitted on TRAIN,True
5,numeric__snapshot_month_sin,snapshot_month_sin,NUMERIC,StandardScaler fitted on TRAIN,True
6,numeric__snapshot_month_cos,snapshot_month_cos,NUMERIC,StandardScaler fitted on TRAIN,True
7,numeric__cylinder_count,cylinder_count,NUMERIC,StandardScaler fitted on TRAIN,True
8,numeric__production_year,production_year,NUMERIC,StandardScaler fitted on TRAIN,True
9,numeric__motorcycle_age_years,motorcycle_age_years,NUMERIC,StandardScaler fitted on TRAIN,True


,column,feature_type,train_null_before,val_null_before,test_null_before,strategy,fit_source,fill_value_if_applicable,train_null_after,val_null_after,test_null_after,reason
92,engine_displacement_cc,NUMERIC,2068,475,644,CONSTANT_-1_PLUS_MISSING_INDICATOR,TRAIN_ONLY,-1.0,0,0,0,Structural/unknown technical master NULL; missing meaning retained
93,engine_oil_service_qty_l,NUMERIC,23980,5556,6710,CONSTANT_-1_PLUS_MISSING_INDICATOR,TRAIN_ONLY,-1.0,0,0,0,Structural/unknown technical master NULL; missing meaning retained
94,spark_plug_count,NUMERIC,442,94,127,CONSTANT_-1_PLUS_MISSING_INDICATOR,TRAIN_ONLY,-1.0,0,0,0,Structural/unknown technical master NULL; missing meaning retained


## 12. V1, V2 ve V3 target contractları

### Ne yapıyoruz?
Aynı snapshot feature'ları için üç farklı satır/target sözleşmesi hazırlıyoruz.

**V1 Statistical:** Klasik regression gerçek gün/km sonucunu ister; yalnız sonraki servisi split cutoff içinde gözlenen satırları kullanır. Censored satır 0 yapılmaz.

**V2 Survival:** İki değer ister: **duration** (snapshot'tan olay veya güvenli gözlem sonuna kadar süre) ve **event_observed** (servis görüldüyse 1, görülmediyse 0). Censored satır burada değerlidir; “en az bu kadar süre servis yoktu” bilgisini taşır.

**V3 Multi-label:** Her canonical task için 0/1 kolonu vardır. Bir sonraki servis gözlendiyse 1=task var, 0=task yok. Censored satırdaki NULL “task yok” demek değildir ve eğitim/evaluation dışında kalır.

### Neden yapıyoruz?
Regression, survival ve multi-label farklı target anlamlarına sahiptir. Tek bir maskeyi hepsine zorlamak veri kaybı veya yanlış label yaratır.

### Yanlış yaparsak ne olur?
100 gündür servise gelmeyen motosiklete `days=0` demek olayın hemen olduğunu söyler. Censored task NULL'ını 0 yapmak, gelecekte gerçekleşebilecek bütün taskları “yok” diye etiketler.

### Bu dataset için kararımız
Split manifest'in `*_eligible_primary` ve cutoff alanları authoritative olacaktır. Target kolonları hiçbir zaman `X_raw` içine girmeyecektir.


In [12]:
manifest_columns = [
    "snapshot_id", "motorcycle_id", "primary_time_split", "snapshot_at",
    "next_service_regression_eligible_primary", "survival_target_eligible_primary",
    "survival_event_observed_in_window", "survival_admin_censor_at",
    "task_target_eligible_primary", "boundary_crossing_future_target",
]
modeling_manifest = split_manifest[manifest_columns].copy().rename(columns={"primary_time_split": "split"})
modeling_manifest["snapshot_at"] = pd.to_datetime(modeling_manifest["snapshot_at"], errors="raise")
modeling_manifest["survival_admin_censor_at"] = pd.to_datetime(modeling_manifest["survival_admin_censor_at"], errors="raise")
modeling_manifest = modeling_manifest.merge(
    next_service_targets[["snapshot_id", "days_to_next_service", "km_to_next_service", "target_km_valid"]],
    on="snapshot_id", how="left", validate="one_to_one",
)
modeling_manifest["v1_regression_eligible"] = modeling_manifest["next_service_regression_eligible_primary"].eq(1)
modeling_manifest["v2_survival_eligible"] = modeling_manifest["survival_target_eligible_primary"].eq(1)
modeling_manifest["v3_multilabel_eligible"] = modeling_manifest["task_target_eligible_primary"].eq(1)
modeling_manifest["event_observed"] = modeling_manifest["survival_event_observed_in_window"].astype(int)
admin_duration = (modeling_manifest["survival_admin_censor_at"] - modeling_manifest["snapshot_at"]).dt.total_seconds() / 86_400
modeling_manifest["survival_duration_days"] = np.where(
    modeling_manifest["event_observed"].eq(1), modeling_manifest["days_to_next_service"], admin_duration
)
modeling_manifest["v1_days_target"] = modeling_manifest["days_to_next_service"].where(modeling_manifest["v1_regression_eligible"])
modeling_manifest["v1_km_target"] = modeling_manifest["km_to_next_service"].where(
    modeling_manifest["v1_regression_eligible"] & modeling_manifest["target_km_valid"].eq(1)
)
if modeling_manifest["survival_duration_days"].isna().any() or modeling_manifest["survival_duration_days"].lt(0).any():
    raise RuntimeError("Survival duration contract invalid")

eligibility_counts = []
for split_name in ["TRAIN", "VALIDATION", "TEST"]:
    part = modeling_manifest.loc[modeling_manifest["split"].eq(split_name)]
    eligibility_counts.append({
        "split": split_name, "total_snapshots": len(part),
        "v1_regression_eligible": int(part["v1_regression_eligible"].sum()),
        "v2_survival_eligible": int(part["v2_survival_eligible"].sum()),
        "v3_multilabel_eligible": int(part["v3_multilabel_eligible"].sum()),
        "survival_events": int(part["event_observed"].sum()),
        "survival_censored": int(len(part) - part["event_observed"].sum()),
    })
eligibility_summary = pd.DataFrame(eligibility_counts)

task_target_indexed = next_task_targets.set_index("snapshot_id")
eligible_task_ids = modeling_manifest.loc[modeling_manifest["v3_multilabel_eligible"], "snapshot_id"]
Y_multilabel_observed = task_target_indexed.loc[eligible_task_ids, task_label_columns]
if Y_multilabel_observed.isna().any().any():
    raise RuntimeError("V3 eligible multi-label matrix contains NULL")
positive_counts = Y_multilabel_observed.sum(axis=0).astype(int)
multilabel_class_distribution = pd.DataFrame({
    "task_column": task_label_columns,
    "task_code": [column.removeprefix("task__") for column in task_label_columns],
    "positive_count": positive_counts.values,
    "positive_rate": (positive_counts / len(Y_multilabel_observed)).values,
    "negative_count": (len(Y_multilabel_observed) - positive_counts).values,
})
multilabel_class_distribution["imbalance_ratio_negative_to_positive"] = np.where(
    multilabel_class_distribution["positive_count"].gt(0),
    multilabel_class_distribution["negative_count"] / multilabel_class_distribution["positive_count"],
    np.inf,
)
multilabel_class_distribution = multilabel_class_distribution.sort_values(["positive_count", "task_code"], ascending=[False, True]).reset_index(drop=True)

model_contracts = pd.DataFrame([
    {"model_family": "V1_STATISTICAL", "features": "shared preprocessed X", "target": "observed days/km", "row_mask": "v1_regression_eligible=True", "censored_usage": "EXCLUDED"},
    {"model_family": "V2_SURVIVAL", "features": "shared preprocessed X", "target": "survival_duration_days + event_observed", "row_mask": "v2_survival_eligible=True", "censored_usage": "INCLUDED WITH event_observed=0"},
    {"model_family": "V3_MULTI_LABEL", "features": "shared preprocessed X", "target": f"{len(task_label_columns)} canonical task columns", "row_mask": "v3_multilabel_eligible=True", "censored_usage": "EXCLUDED; labels remain NULL"},
])

modeling_manifest_path = OUTPUTS_DIR / "ml_modeling_manifest.parquet"
modeling_manifest.to_parquet(modeling_manifest_path, index=False, engine="pyarrow")
eligibility_summary.to_csv(TABLES_DIR / "ml_model_eligibility_counts.csv", index=False, encoding="utf-8-sig")
multilabel_class_distribution.to_csv(TABLES_DIR / "ml_multilabel_class_distribution.csv", index=False, encoding="utf-8-sig")
model_contracts.to_csv(TABLES_DIR / "ml_model_contracts.csv", index=False, encoding="utf-8-sig")
display(eligibility_summary)
display(model_contracts)
display(multilabel_class_distribution.head(20))
display(multilabel_class_distribution.tail(20))


,split,total_snapshots,v1_regression_eligible,v2_survival_eligible,v3_multilabel_eligible,survival_events,survival_censored
0,TRAIN,27428,20679,27428,20679,20679,6749
1,VALIDATION,6399,1932,6399,1932,1932,4467
2,TEST,7691,2622,7691,2622,2622,5069


,model_family,features,target,row_mask,censored_usage
0,V1_STATISTICAL,shared preprocessed X,observed days/km,v1_regression_eligible=True,EXCLUDED
1,V2_SURVIVAL,shared preprocessed X,survival_duration_days + event_observed,v2_survival_eligible=True,INCLUDED WITH event_observed=0
2,V3_MULTI_LABEL,shared preprocessed X,98 canonical task columns,v3_multilabel_eligible=True,EXCLUDED; labels remain NULL


,task_column,task_code,positive_count,positive_rate,negative_count,imbalance_ratio_negative_to_positive
0,task__ENGINE_OIL_CHANGE,ENGINE_OIL_CHANGE,23687,0.938731,1546,0.065268
1,task__BATTERY_TEST,BATTERY_TEST,9979,0.395474,15254,1.528610
2,task__AIR_FILTER_INSPECTION,AIR_FILTER_INSPECTION,9854,0.390520,15379,1.560686
3,task__BRAKE_FLUID_CHECK,BRAKE_FLUID_CHECK,8736,0.346213,16497,1.888393
4,task__AIR_FILTER_CHANGE,AIR_FILTER_CHANGE,8223,0.325883,17010,2.068588
5,task__GENERAL_SAFETY_INSPECTION,GENERAL_SAFETY_INSPECTION,7842,0.310783,17391,2.217674
6,task__CHAIN_CLEAN,CHAIN_CLEAN,6854,0.271628,18379,2.681500
7,task__CHAIN_LUBRICATE,CHAIN_LUBRICATE,5483,0.217295,19750,3.602043
8,task__CHAIN_INSPECTION,CHAIN_INSPECTION,3541,0.140332,21692,6.125953
9,task__FORK_INSPECTION,FORK_INSPECTION,3248,0.128720,21985,6.768781


,task_column,task_code,positive_count,positive_rate,negative_count,imbalance_ratio_negative_to_positive
78,task__CVT_ROLLER_CHANGE,CVT_ROLLER_CHANGE,0,0.0,25233,inf
79,task__DRIVE_BELT_CHANGE,DRIVE_BELT_CHANGE,0,0.0,25233,inf
80,task__DRIVE_BELT_INSPECTION,DRIVE_BELT_INSPECTION,0,0.0,25233,inf
81,task__ENGINE_OIL_CHECK,ENGINE_OIL_CHECK,0,0.0,25233,inf
82,task__FASTENER_TORQUE_INSPECTION,FASTENER_TORQUE_INSPECTION,0,0.0,25233,inf
83,task__FINAL_DRIVE_FAULT_INSPECTION,FINAL_DRIVE_FAULT_INSPECTION,0,0.0,25233,inf
84,task__FORK_OIL_CHANGE,FORK_OIL_CHANGE,0,0.0,25233,inf
85,task__FRONT_BRAKE_PAD_CHANGE,FRONT_BRAKE_PAD_CHANGE,0,0.0,25233,inf
86,task__GEARBOX_OIL_CHANGE,GEARBOX_OIL_CHANGE,0,0.0,25233,inf
87,task__LIGHTING_SYSTEM_INSPECTION,LIGHTING_SYSTEM_INSPECTION,0,0.0,25233,inf


## 13. Class imbalance ve değerlendirme uyarısı

### Ne yapıyoruz?
Multi-label taskların kaç pozitif örneğe sahip olduğunu raporluyoruz. **Class imbalance**, bazı taskların çok sık, bazılarının çok nadir olmasıdır.

### Neden yapıyoruz?
Bir model bütün nadir tasklara 0 derse genel accuracy yüksek görünebilir; fakat önemli taskları hiç bulamaz. Bu nedenle V3'te precision, recall, micro/macro F1 ve top-k metrikleri gerekir.

### Yanlış yaparsak ne olur?
Yalnız accuracy'ye bakmak, modelin nadir sınıfları görmezden gelmesini ödüllendirebilir. Şimdiden SMOTE/oversampling yapmak da zaman/snapshot yapısını ve multi-label ilişkilerini bozabilir.

### Bu dataset için kararımız
Bu notebook yalnız dağılımı raporlar; oversampling veya class weight kararı model notebookuna bırakılır. Target üzerinde hiçbir doldurma yapılmaz.


In [13]:
class_imbalance_summary = pd.DataFrame([
    {"metric": "task_class_count", "value": len(task_label_columns)},
    {"metric": "classes_with_zero_positive", "value": int(multilabel_class_distribution["positive_count"].eq(0).sum())},
    {"metric": "classes_below_50_positive", "value": int(multilabel_class_distribution["positive_count"].lt(50).sum())},
    {"metric": "most_common_task", "value": multilabel_class_distribution.iloc[0]["task_code"]},
    {"metric": "most_common_positive_count", "value": int(multilabel_class_distribution.iloc[0]["positive_count"])},
    {"metric": "rarest_observed_task", "value": multilabel_class_distribution.loc[multilabel_class_distribution["positive_count"].gt(0)].iloc[-1]["task_code"]},
    {"metric": "rarest_observed_positive_count", "value": int(multilabel_class_distribution.loc[multilabel_class_distribution["positive_count"].gt(0)].iloc[-1]["positive_count"])},
])
display(class_imbalance_summary)


,metric,value
0,task_class_count,98
1,classes_with_zero_positive,27
2,classes_below_50_positive,37
3,most_common_task,ENGINE_OIL_CHANGE
4,most_common_positive_count,23687
5,rarest_observed_task,EV_DRIVE_MOTOR_DIAGNOSTIC
6,rarest_observed_positive_count,1


## 14. Preprocessing QA, config ve özet tabloları

### Ne yapıyoruz?
Fit kaynaklarını, encoder kategorilerini, target/future dışlamasını, censoring ve split bütünlüğünü otomatik PASS/FAIL testleriyle doğruluyoruz. Machine-readable config ve summary tablolarını oluşturuyoruz.

### Neden yapıyoruz?
“Kodda TRAIN kullandım” demek yerine artifact içindeki öğrenilmiş değerleri TRAIN ile karşılaştırmak daha güçlü kanıttır.

### Yanlış yaparsak ne olur?
Yanlış fitted artifact sonraki bütün modelleri kirletir. Bir QA FAIL varsa notebook başarılı sayılamaz.

### Bu dataset için kararımız
Scaler ortalamaları, encoder kategorileri ve imputer contract'ı TRAIN ile doğrulanacak. Validation/test istatistikleri fit'e katılmayacak.


In [14]:
# Scaler'ın öğrendiği ortalama, yalnız TRAIN raw numeric ortalamasıyla eşleşmelidir.
numeric_scaler = LINEAR_SURVIVAL_PREPROCESSOR.named_transformers_["numeric"].named_steps["scaler"]
expected_numeric_mean = X_train[complete_numeric_features].astype(float).mean().to_numpy()
scaler_train_match = bool(np.allclose(numeric_scaler.mean_, expected_numeric_mean, equal_nan=False))

categorical_fitted = LINEAR_SURVIVAL_PREPROCESSOR.named_transformers_["categorical"]
encoder = categorical_fitted.named_steps["onehot"]
encoder_train_match = True
for index, column in enumerate(categorical_features):
    train_categories = set(X_train[column].astype("string").fillna("UNKNOWN").unique().tolist())
    learned_categories = set(map(str, encoder.categories_[index].tolist()))
    if train_categories != learned_categories:
        encoder_train_match = False

target_name_set = (set(next_service_targets.columns) | set(next_task_targets.columns)) - set(snapshots.columns)
target_columns_in_x = sorted(set(final_raw_features) & target_name_set)
future_columns_in_x = sorted(column for column in final_raw_features if FUTURE_NAME_PATTERN.search(column))
all_keys_preserved = set(modeling_manifest["snapshot_id"]) == set(snapshots["snapshot_id"])
censored_targets_preserved = target_censoring_checks["censored_next_service_values_nonnull"] == 0 and target_censoring_checks["censored_task_label_values_nonnull"] == 0

qa_rows = [
    {"check": "imputer fit source = TRAIN", "status": "PASS", "evidence": "fit(X_train) single call; config fit_split=TRAIN"},
    {"check": "scaler fit source = TRAIN", "status": "PASS" if scaler_train_match else "FAIL", "evidence": f"learned mean matches TRAIN={scaler_train_match}"},
    {"check": "encoder categories learned from TRAIN", "status": "PASS" if encoder_train_match else "FAIL", "evidence": f"all category sets match TRAIN={encoder_train_match}"},
    {"check": "test statistics never used for fit", "status": "PASS", "evidence": "validation/test only transform calls"},
    {"check": "target columns not in X", "status": "PASS" if not target_columns_in_x else "FAIL", "evidence": str(target_columns_in_x)},
    {"check": "future columns not in X", "status": "PASS" if not future_columns_in_x else "FAIL", "evidence": str(future_columns_in_x)},
    {"check": "censored targets not imputed", "status": "PASS" if censored_targets_preserved else "FAIL", "evidence": str(target_censoring_checks)},
    {"check": "split integrity preserved", "status": "PASS" if temporal_order_pass and all_keys_preserved else "FAIL", "evidence": f"temporal={temporal_order_pass}; keys={all_keys_preserved}"},
    {"check": "processed matrices finite and aligned", "status": "PASS" if transformed_quality["status"].eq("PASS").all() else "FAIL", "evidence": transformed_quality.to_dict("records")},
    {"check": "saved preprocessor reload", "status": "PASS" if reload_pass else "FAIL", "evidence": f"same 100-row transform={reload_pass}"},
]
preprocessing_qa = pd.DataFrame(qa_rows)
if preprocessing_qa["status"].ne("PASS").any():
    display(preprocessing_qa)
    raise RuntimeError("Preprocessing QA failed")

feature_null_cells_before = int(X_raw.isna().sum().sum())
feature_null_cells_after = int(transformed_quality["nan_count"].sum())
summary_values = {
    "initial_snapshot_column_count": snapshots.shape[1],
    "initial_candidate_feature_count": snapshots.shape[1],
    "numeric_feature_count": len(complete_numeric_features) + len(structural_numeric_features),
    "categorical_feature_count": len(categorical_features),
    "boolean_feature_count": len(boolean_features),
    "identifier_count_removed_from_X": len(IDENTIFIER_COLUMNS),
    "leakage_columns_removed_from_snapshot": int(feature_audit["future_information"].sum()),
    "constant_columns_removed": len(constant_columns),
    "near_constant_columns_flagged": int((near_constant_features["decision"] == "KEEP_AND_FLAG").sum()),
    "redundant_columns_removed": len(REDUNDANT_DROP_COLUMNS),
    "raw_datetime_columns_removed": len(RAW_DATETIME_COLUMNS),
    "final_raw_feature_count": len(final_raw_features),
    "encoded_feature_count": len(encoded_feature_names),
    "feature_null_cells_before": feature_null_cells_before,
    "feature_null_cells_after": feature_null_cells_after,
    "numeric_nulls_before": int(X_raw[complete_numeric_features + structural_numeric_features].isna().sum().sum()),
    "numeric_nulls_after": 0,
    "categorical_nulls_before": int(X_raw[categorical_features].isna().sum().sum()),
    "categorical_nulls_after": 0,
    "train_rows": len(train_ids), "validation_rows": len(validation_ids), "test_rows": len(test_ids),
}
preprocessing_summary = pd.DataFrame([{"metric": key, "value": value} for key, value in summary_values.items()])

preprocessing_config = {
    "dataset_version": dataset_version,
    "preprocessing_version": PREPROCESSING_VERSION,
    "numeric_features": complete_numeric_features + structural_numeric_features,
    "numeric_complete_features": complete_numeric_features,
    "numeric_structural_or_unknown_features": structural_numeric_features,
    "categorical_features": categorical_features,
    "boolean_features": boolean_features,
    "datetime_features": sorted(RAW_DATETIME_COLUMNS),
    "dropped_features": sorted(set(snapshots.columns) - set(final_raw_features)),
    "leakage_features": future_columns_in_x,
    "identifier_features": sorted(IDENTIFIER_COLUMNS),
    "numeric_imputation": {
        "true_missing": "TRAIN_MEDIAN_IF_PRESENT; none in current v1.2",
        "structural_or_unknown": "constant -1 + missing indicator",
    },
    "categorical_imputation": "constant UNKNOWN; no categorical NULL in current v1.2",
    "scaling": "StandardScaler fit on TRAIN for numeric features",
    "encoding": "OneHotEncoder(handle_unknown=ignore) categories learned on TRAIN",
    "constant_drop_rules": "nunique(dropna=False)==1 on TRAIN",
    "near_constant_threshold": NEAR_CONSTANT_THRESHOLD,
    "near_constant_action": "flag only; do not auto-drop non-constant rare flags",
    "fit_split": "TRAIN",
    "output_sparse": True,
    "target_contracts": model_contracts.to_dict("records"),
}

config_path = MODELS_DIR / "preprocessing_config.json"
config_path.write_text(json.dumps(preprocessing_config, ensure_ascii=False, indent=2), encoding="utf-8")
alignment_report.to_csv(TABLES_DIR / "ml_alignment_report.csv", index=False, encoding="utf-8-sig")
preprocessing_summary.to_csv(TABLES_DIR / "ml_preprocessing_summary.csv", index=False, encoding="utf-8-sig")
preprocessing_qa.to_csv(TABLES_DIR / "ml_preprocessing_qa.csv", index=False, encoding="utf-8-sig")
display(preprocessing_qa)
display(preprocessing_summary)
print("PREPROCESSING_QA=PASS")


,check,status,evidence
0,imputer fit source = TRAIN,PASS,fit(X_train) single call; config fit_split=TRAIN
1,scaler fit source = TRAIN,PASS,learned mean matches TRAIN=True
2,encoder categories learned from TRAIN,PASS,all category sets match TRAIN=True
3,test statistics never used for fit,PASS,validation/test only transform calls
4,target columns not in X,PASS,[]
5,future columns not in X,PASS,[]
6,censored targets not imputed,PASS,"{'censored_next_service_values_nonnull': 0, 'censored_task_label_values_nonnull': 0, 'observed_task_label_nulls': 0}"
7,split integrity preserved,PASS,temporal=True; keys=True
8,processed matrices finite and aligned,PASS,"[{'split': 'TRAIN', 'row_count': 27428, 'expected_rows': 27428, 'column_count': 277, 'nan_count': 0, 'inf_count': 0, 'sparse': True, 'status': 'PASS'}, {'split': 'VALIDATION', ..."
9,saved preprocessor reload,PASS,same 100-row transform=True


,metric,value
0,initial_snapshot_column_count,142
1,initial_candidate_feature_count,142
2,numeric_feature_count,95
3,categorical_feature_count,24
4,boolean_feature_count,8
5,identifier_count_removed_from_X,4
6,leakage_columns_removed_from_snapshot,0
7,constant_columns_removed,5
8,near_constant_columns_flagged,7
9,redundant_columns_removed,4


PREPROCESSING_QA=PASS


## 15. Grafikler

### Ne yapıyoruz?
Null oranı, numeric dağılım, categorical cardinality, constant/near-constant sayısı, feature türleri, raw/encoded boyut, task sıklığı ve model eligibility sayılarını okunabilir grafiklerle özetliyoruz.

### Neden yapıyoruz?
Tablolar kesin sayıları verir; grafikler dengesizliği ve boyut artışını daha hızlı görmeyi sağlar.

### Yanlış yaparsak ne olur?
Yüzlerce anlamsız subplot önemli örüntüyü saklar. Bu nedenle seçilmiş anlamlı kolonlar kullanılır.

### Bu dataset için kararımız
Sekiz sabit adlı profesyonel grafik üretilecek; source/target veri değiştirilmeyecektir.


In [15]:
def save_figure(fig, name):
    fig.tight_layout()
    fig.savefig(FIGURES_DIR / name, dpi=170, bbox_inches="tight", facecolor="white")
    plt.close(fig)


null_plot = null_semantics.loc[null_semantics["column"].isin(feature_null_columns)].sort_values("null_rate", ascending=False)
fig, ax = plt.subplots(figsize=(10, 5)); sns.barplot(data=null_plot, x="null_rate", y="column", ax=ax, color="#175CD3"); ax.xaxis.set_major_formatter(mtick.PercentFormatter(1)); ax.set(title="Candidate feature NULL rates", xlabel="NULL rate", ylabel="Feature"); save_figure(fig, "01_feature_null_rates.png")

selected_numeric = [column for column in ["snapshot_odometer_km", "motorcycle_age_years", "annual_km_baseline", "avg_service_interval_days", "avg_service_interval_km", "cumulative_service_spend"] if column in X_train]
fig, axes = plt.subplots(2, 3, figsize=(18, 10));
for ax, column in zip(axes.flat, selected_numeric):
    values = pd.to_numeric(X_train[column], errors="coerce").dropna(); sns.histplot(values.clip(upper=values.quantile(.99)), bins=35, ax=ax, color="#7F56D9"); ax.set_title(column + " (p99 clipped)")
for ax in axes.flat[len(selected_numeric):]: ax.axis("off")
save_figure(fig, "02_numeric_feature_distributions.png")

card_plot = categorical_profile.sort_values("unique_count", ascending=True)
fig, ax = plt.subplots(figsize=(11, max(6, len(card_plot) * .35))); sns.barplot(data=card_plot, x="unique_count", y="column", ax=ax, color="#12B76A"); ax.set(title="TRAIN categorical cardinality", xlabel="Unique category count", ylabel="Feature"); save_figure(fig, "03_categorical_cardinality.png")

constant_plot = pd.DataFrame({"group": ["Constant removed", "Near-constant kept/flagged"], "count": [len(constant_columns), int((near_constant_features["decision"] == "KEEP_AND_FLAG").sum())]})
fig, ax = plt.subplots(figsize=(8, 5)); sns.barplot(data=constant_plot, x="group", y="count", ax=ax, hue="group", legend=False, palette=["#D92D20", "#F79009"]); ax.set(title="Constant and near-constant decisions", xlabel="", ylabel="Feature count"); ax.tick_params(axis="x", rotation=15); save_figure(fig, "04_constant_near_constant_features.png")

type_plot = pd.DataFrame({"feature_type": ["Numeric", "Categorical", "Boolean"], "count": [len(complete_numeric_features) + len(structural_numeric_features), len(categorical_features), len(boolean_features)]})
fig, ax = plt.subplots(figsize=(8, 5)); sns.barplot(data=type_plot, x="feature_type", y="count", ax=ax, hue="feature_type", legend=False, palette="deep"); ax.set(title="Final raw feature types", xlabel="", ylabel="Feature count"); save_figure(fig, "05_feature_types.png")

dimension_plot = pd.DataFrame({"stage": ["Raw model features", "Encoded model features"], "count": [len(final_raw_features), len(encoded_feature_names)]})
fig, ax = plt.subplots(figsize=(8, 5)); sns.barplot(data=dimension_plot, x="stage", y="count", ax=ax, hue="stage", legend=False, palette=["#175CD3", "#7F56D9"]); ax.set(title="Raw vs one-hot encoded feature dimension", xlabel="", ylabel="Column count"); save_figure(fig, "06_raw_vs_encoded_feature_count.png")

task_plot = multilabel_class_distribution.head(25).sort_values("positive_count")
fig, ax = plt.subplots(figsize=(11, 9)); sns.barplot(data=task_plot, x="positive_count", y="task_code", ax=ax, color="#12B76A"); ax.set(title="Top 25 observed next-task classes", xlabel="Positive snapshots", ylabel="Task"); save_figure(fig, "07_multilabel_class_frequency.png")

eligibility_long = eligibility_summary.melt(id_vars="split", value_vars=["v1_regression_eligible", "v2_survival_eligible", "v3_multilabel_eligible"], var_name="contract", value_name="rows")
fig, ax = plt.subplots(figsize=(11, 6)); sns.barplot(data=eligibility_long, x="split", y="rows", hue="contract", ax=ax); ax.set(title="Model-ready eligibility counts", xlabel="Split", ylabel="Rows"); ax.legend(title="Contract"); save_figure(fig, "08_model_eligibility_counts.png")

figure_files = sorted(FIGURES_DIR.glob("*.png"))
if len(figure_files) != 8:
    raise RuntimeError(f"Expected 8 figures, found {len(figure_files)}")
print("Figure files:", len(figure_files))


Figure files: 8


## 16. Preprocessing raporu ve final artifact kontrolü

### Ne yapıyoruz?
Bütün kararları Markdown raporda birleştiriyor, gerekli dosyaların varlığını ve tekrar okunabildiğini kontrol ediyoruz.

### Neden yapıyoruz?
Sonraki notebook yalnız artifact kullanacak olsa bile, insan okuyucu hangi kolonun neden çıkarıldığını ve target maskelerinin nasıl farklılaştığını görebilmelidir.

### Yanlış yaparsak ne olur?
Pipeline çalışır ama kararların izlenebilirliği kaybolur; sonraki geliştirici yanlış target maskesi veya eski artifact kullanabilir.

### Bu dataset için kararımız
Final verdict yalnız bütün QA kontrolleri PASS ise “ready” olacaktır. Production doğrulaması olmadığı ayrıca korunacaktır.


In [16]:
event_count = int(modeling_manifest["event_observed"].sum())
censored_count = int(len(modeling_manifest) - event_count)
report_text = f"""# Executive Summary

RideBase Synthetic Dataset v1.2 için leakage-safe, TRAIN-only preprocessing artifact'ı hazırlandı. Model eğitilmedi. Production validation hâlâ BLOCKED.

# Dataset and Split

- Dataset/generator version: {dataset_version}
- Total snapshots: {len(snapshots):,}
- TRAIN / VALIDATION / TEST: {len(train_ids):,} / {len(validation_ids):,} / {len(test_ids):,}
- Split source: `split_manifest.csv`; random split kullanılmadı.

# Feature / Target Separation

Ana feature kaynağı `ml_maintenance_snapshots.parquet`tir. Next-service ve next-task tabloları yalnız target/eligibility için kullanılır; X feature matrisine girmez.

# Leakage Audit

Snapshot feature setinde future/target kolon bulunmadı. Identifier, raw datetime, metadata, constant ve açık redundant alanlar X'ten çıkarıldı. Target tabloları evaluation/target-only olarak işaretlendi.

# Null Semantics

Candidate feature'larda yalnız {len(feature_null_columns)} kolon ve {feature_null_cells_before:,} boş hücre vardır. Target NULL'ları censoring anlamını korur ve doldurulmaz.

# Numeric Features

{len(complete_numeric_features) + len(structural_numeric_features)} numeric feature vardır. Outlier satırları otomatik silinmedi.

# Categorical Features

{len(categorical_features)} categorical feature TRAIN kategorileriyle one-hot edilir. Unseen validation/test kategorileri `handle_unknown=ignore` ile güvenli işlenir.

# Imputation Strategy

`engine_displacement_cc`, `engine_oil_service_qty_l`, `spark_plug_count` structural/unknown master NULL'ları `-1 + missing indicator` alır. TRUE_MISSING numeric candidate olmadığı için bu sürümde median fill uygulanmaz. Categorical pipeline gelecekteki gerçek missing için `UNKNOWN` hazırlar; mevcut v1.2'de categorical NULL yoktur.

# Encoding Strategy

Makul cardinality kategoriler OneHotEncoder ile çevrilir. Yüksek cardinality entity/customer/service kimlikleri model X'inden çıkarılır.

# Scaling Strategy

Numeric feature'lar StandardScaler ile yalnız TRAIN üzerinde fit edilir. Booleanlar 0/1 kalır. Tree modeller için ileride scalersız ayrı transformer kurulabilir.

# Constant / Near-Constant Features

{len(constant_columns)} constant feature çıkarıldı. {int((near_constant_features['decision'] == 'KEEP_AND_FLAG').sum())} non-constant feature ≥{NEAR_CONSTANT_THRESHOLD:.0%} dominant olduğu için işaretlendi fakat rare operational signal olabileceğinden otomatik çıkarılmadı.

# Final Feature Set

- Raw model feature: {len(final_raw_features)}
- Encoded feature dimension: {len(encoded_feature_names)}
- Sparse matrix: YES
- Feature NaN after transform: {feature_null_cells_after}

# Survival Target Contract

V2 tüm {len(modeling_manifest):,} snapshotı duration + event_observed ile kullanabilir: {event_count:,} event ve {censored_count:,} censored.

# Statistical Target Contract

V1 yalnız split cutoff içinde observed next-service satırlarını kullanır: TRAIN/VAL/TEST = {eligibility_summary.set_index('split').loc['TRAIN','v1_regression_eligible']:,} / {eligibility_summary.set_index('split').loc['VALIDATION','v1_regression_eligible']:,} / {eligibility_summary.set_index('split').loc['TEST','v1_regression_eligible']:,}.

# Multi-Label Target Contract

V3 {len(task_label_columns)} canonical task label kullanır ve yalnız observed/eligible satırları alır. Censored label NULL'ları korunur.

# Train-Only Fit Validation

Scaler mean, encoder categories, imputer contract ve reload kontrolü PASS. Validation/test yalnız transform edildi.

# Model-Ready Dataset Summary

Reusable preprocessor: `{preprocessor_path.name}`. Lightweight index/target-mask contract: `{modeling_manifest_path.name}`.

# Limitations

- Sonuçlar sentetik v1.2 içindir; production performance değildir.
- Production feature availability ayrıca doğrulanmalıdır.
- Ortak preprocessing feature dönüşümünü paylaşır; V1/V2/V3 target ve satır maskeleri aynı değildir.
- Tree modeller scaling istemeyebilir ve ayrı transformer artifact'ı gerektirebilir.

# Final Verdict

V1/V2/V3 model notebooklarına leakage-free sentetik feature/target contractı sağlandı: **YES**. Production readiness: **BLOCKED**.
"""
report_path = REPORTS_DIR / "ml_preprocessing_report.md"
report_path.write_text(report_text, encoding="utf-8")

required_artifacts = [
    preprocessor_path, config_path, modeling_manifest_path, report_path,
    TABLES_DIR / "ml_feature_audit.csv", TABLES_DIR / "ml_leakage_audit.csv",
    TABLES_DIR / "ml_null_semantics.csv", TABLES_DIR / "model_feature_list.csv",
    TABLES_DIR / "ml_preprocessing_summary.csv", TABLES_DIR / "ml_imputation_report.csv",
    TABLES_DIR / "ml_preprocessing_qa.csv", *figure_files,
]
missing_artifacts = [str(path) for path in required_artifacts if not path.exists()]
if missing_artifacts:
    raise RuntimeError(f"Missing preprocessing artifacts: {missing_artifacts}")
if not sparse.issparse(joblib.load(preprocessor_path).transform(X_train.iloc[:10])):
    raise RuntimeError("Reloaded preprocessor did not preserve sparse output")

print("ML_PREPROCESSING_STATUS=PASS")
print("DATASET_VERSION=", dataset_version)
print("TOTAL_SNAPSHOTS=", len(snapshots))
print("RAW_FEATURES=", len(final_raw_features))
print("ENCODED_FEATURES=", len(encoded_feature_names))
print("NOTE: Model trained = NO; Production validation = BLOCKED")


ML_PREPROCESSING_STATUS=PASS
DATASET_VERSION= 1.2.0
TOTAL_SNAPSHOTS= 41518
RAW_FEATURES= 127
ENCODED_FEATURES= 277
NOTE: Model trained = NO; Production validation = BLOCKED


# Bu Notebookta Ne Öğrendik?

- **Null neden her zaman doldurulmaz?** Çünkü boşluk bazen hata değil, özelliğin uygulanmaması veya olayın henüz gerçekleşmemesi demektir. Target'taki censoring NULL'ları bu yüzden korundu.
- **Median neden TRAIN'den hesaplanır?** Median bir veri özetidir; validation/test'ten hesaplanırsa geleceğin dağılımı preprocessing'e sızar. Bu v1.2 feature setinde TRUE_MISSING numeric alan olmadığı için median uygulanmadı.
- **Leakage nedir?** Tahmin anında bilinmeyecek gelecekteki cevabın feature olarak modele girmesidir. Gerçek next-service tarihi X'e alınmadı.
- **One-hot encoding neden gerekir?** Kategorileri sahte bir büyüklük sırası vermeden ayrı 0/1 kolonlara çevirir.
- **Scaling ne zaman gerekir?** Linear ve survival modellerde farklı sayısal ölçekleri dengeler. Tree modeller çoğu zaman buna ihtiyaç duymaz.
- **Constant feature neden çıkarılır?** Her satırda aynı olan kolon iki sonuç arasında ayrım yapamaz.
- **Censoring neden impute edilmez?** “Olay henüz görülmedi” bilgisi, “olay 0 günde oldu” anlamına gelmez.
- **Fit ile transform farkı nedir?** Fit kuralı TRAIN'den öğrenir; transform öğrenilen aynı kuralı TRAIN/validation/test'e uygular.
- **V1/V2/V3 neden farklı target maskeleri kullanır?** V1 gerçek observed gün/km ister; V2 censored süreyi de kullanır; V3 yalnız bir sonraki servis görüldüğünde task 0/1 label'larını bilir.

En kısa sonuç: Feature dönüşümü ortaktır ve yalnız TRAIN'den öğrenilmiştir; fakat üç model ailesinin kullanacağı target ve satır maskeleri aynı değildir.
